In [ ]:
from requirements_deprecated import *

# 1. 

In [ ]:
filename = "h2_method_comparison.pkl"
raw_data = load_data(filename) 

distances = np.concatenate([np.linspace(0.3, 1.0, 15), np.linspace(1.1, 2.5, 5)])
distances = np.sort(distances)

records = []
if raw_data:
    for method, energy_list in raw_data.items():
        for i, energy in enumerate(energy_list):
            if energy is None or np.isnan(energy):
                continue
                
            dist = distances[i] if i < len(distances) else distances[-1]
            records.append({
                "Method": method,
                "Distance": dist,
                "Energy": energy
            })
    df_methods = pd.DataFrame(records)
else:
    raise ValueError("Erro")

In [ ]:
def plot_method_comparison(df, config, visuals, use_visuals=False, save_image=False):
    methods = [m for m in df["Method"].unique() if m != "PySCF_FCI"]
    
    color_map = {
        "PySCF_HF": "#d62728",      
        "PySCF_DFT": "#ff7f0e",     
        "PySCF_FCI": "#000000",     
        "Qiskit_Exact": "#2ca02c",  
        "Qiskit_VQE": "#1f77b4"     
    }
    
    base_marker_map = {
        "PySCF_HF": "X",
        "PySCF_DFT": "s",
        "PySCF_FCI": "",    
        "Qiskit_Exact": "D", 
        "Qiskit_VQE": "o"
    }

    label_map = {
        "PySCF_HF": "HF",           
        "PySCF_DFT": "DFT",
        "PySCF_FCI": "FCI",
        "Qiskit_Exact": "VQE Exact",
        "Qiskit_VQE": "VQE Sampling"
    }

    fig, axes = plt.subplots(1, len(config["panels"]), figsize=(18, 5))
    if len(config["panels"]) == 1: axes = [axes]
    plt.subplots_adjust(wspace=config["wspace"])
    
    fci_data_df = df[df["Method"] == "PySCF_FCI"].sort_values("Distance")
    fci_dist = fci_data_df["Distance"].values
    fci_energies = fci_data_df["Energy"].values

    def morse_func(r, De, a, re, E_shift):
        return De * (1 - np.exp(-a * (r - re)))**2 + E_shift

    def get_plot_curve(x, y, fit_type, cfg):
        idx = np.argsort(x)
        x, y = x[idx], y[idx]
        
        if fit_type == "raw": return x, y
        
        x_dense = np.linspace(min(x), max(x), 500)
        
        if fit_type == "spline":
            try: return x_dense, make_interp_spline(x, y, k=3)(x_dense)
            except: return x, y
            
        elif fit_type == "morse":
            try:
                mask = x > 0.45 
                p0 = [max(y[mask]) - min(y[mask]), 2.0, x[np.argmin(y)], min(y[mask])]
                popt, _ = curve_fit(morse_func, x[mask], y[mask], p0=p0, maxfev=20000)
                return x_dense, morse_func(x_dense, *popt)
            except: return x, y
            
        elif fit_type == "poly":
            interval = cfg.get("fit_interval", (0.5, 1.0))
            mask = (x >= interval[0]) & (x <= interval[1])
            if len(x[mask]) > cfg.get("poly_degree", 4):
                coeffs = np.polyfit(x[mask], y[mask], cfg.get("poly_degree", 4))
                x_local = np.linspace(interval[0], interval[1], 300)
                return x_local, np.poly1d(coeffs)(x_local)
            return x, y
            
        return x, y 

    fci_min_x, fci_min_y = None, None
    if len(fci_dist) > 0:
        zoom_fit = config.get("fit_methods", {}).get("energy_zoomed", "spline")
        f_x, f_y = get_plot_curve(fci_dist, fci_energies, zoom_fit, config)
        f_min_idx = np.argmin(f_y)
        fci_min_x, fci_min_y = f_x[f_min_idx], f_y[f_min_idx]

    title_map = {0: "(a)", 1: "(b)", 2: "(c)"}

    for idx, panel in enumerate(config["panels"]):
        ax = axes[idx]
        p_cfg = visuals.get(panel, {}) if use_visuals else {}
        
        fit_method = config.get("fit_methods", {}).get(panel, "raw")
        
        if panel == "energy_raw" or panel == "energy_zoomed":
            for m_idx, method in enumerate(["PySCF_FCI"] + methods):
                d = df[df["Method"] == method].sort_values("Distance")
                if len(d) == 0: continue
                
                c = color_map.get(method, "grey")
                mk = base_marker_map.get(method, "o")
                
                ls_dict = p_cfg.get("line_styles")
                ls = ls_dict.get(method, "-" if method != "PySCF_FCI" else "--") if ls_dict else ("-" if method != "PySCF_FCI" else "--")
                show_mk = p_cfg.get("show_markers", True) and method != "PySCF_FCI"
                
                ms = config.get("marker_size", 50)
                mh_dict = p_cfg.get("marker_hierarchy")
                if mh_dict is not None: ms = mh_dict.get(method, ms)
                if panel == "energy_zoomed": ms *= 1.5
                
                alpha_val = config.get("alpha", 0.85)
                ah_dict = p_cfg.get("alpha_hierarchy")
                if ah_dict is not None and method != "PySCF_FCI": alpha_val = ah_dict.get(method, alpha_val)
                if method == "PySCF_FCI": alpha_val = config.get("alpha", 0.85)
                
                x_raw, y_raw = d["Distance"].values, d["Energy"].values
                x_curve, y_curve = get_plot_curve(x_raw, y_raw, fit_method, config)
                
                dec_type = p_cfg.get("decimation")
                mask = np.ones(len(x_raw), dtype=bool)
                if dec_type == "interleaved" and method != "PySCF_FCI":
                    stride = p_cfg.get("stride", 3)
                    mask = (np.arange(len(x_raw)) % stride) == (m_idx % stride)
                elif dec_type == "density" and method != "PySCF_FCI":
                    dy = np.abs(np.gradient(y_raw))
                    mask = (dy > p_cfg.get("density_threshold", 0.05)) | (np.arange(len(x_raw)) % p_cfg.get("sparse_stride", 4) == 0)

                lw = 2.5 if method == "PySCF_FCI" else config["line_width"] 
                ax.plot(x_curve, y_curve, color=c, linestyle=ls, lw=lw, alpha=alpha_val, zorder=2)
                
                if show_mk: 
                    ax.scatter(x_raw[mask], y_raw[mask], color=c, marker=mk, 
                               s=ms, edgecolors=config["outline_color"], 
                               linewidths=config["outline_width"], zorder=3, alpha=alpha_val)
                
            if panel == "energy_zoomed" and fci_min_x is not None:
                if config.get("show_fci_min_line", False):
                    ax.axvline(x=fci_min_x, color='black', linestyle=':', alpha=0.5, zorder=1)
                if config.get("show_fci_star", False):
                    ax.scatter(fci_min_x, fci_min_y, color='red', marker='*', 
                               s=config.get("marker_size", 50) * 6, edgecolors='none', zorder=10)

            ax.set_title(title_map[idx], loc='center')
            ax.set_ylabel("Energy (Ha)")
            ax.set_xlabel(r"Distance $d$ ($\AA$)")
            ax.grid(False)
            
            if panel == "energy_raw":
                ax.set_xlim(config.get("raw_xlim", (0.25, 2.6)))
                ax.set_ylim(config.get("raw_ylim", (-1.2, -0.4)))
            elif panel == "energy_zoomed":
                ax.set_xlim(config.get("zoom_window", (0.5, 1.0)))
                ax.set_ylim(config.get("zoom_ylim", (-1.18, -1.06)))
                
        elif panel == "error_fci":
            for m_idx, method in enumerate(methods):
                d = df[df["Method"] == method].sort_values("Distance")
                if len(d) == 0: continue
                
                fci_interp = np.interp(d["Distance"].values, fci_dist, fci_energies)
                error = np.abs(d["Energy"].values - fci_interp)
                
                c = color_map.get(method, "grey")
                mk = base_marker_map.get(method, "o")
                
                ls_dict = p_cfg.get("line_styles")
                ls = ls_dict.get(method, "-") if ls_dict else "-"
                
                show_mk = p_cfg.get("show_markers", True)
                ms = config.get("marker_size", 50)
                mh_dict = p_cfg.get("marker_hierarchy")
                if mh_dict is not None: ms = mh_dict.get(method, ms)
                
                alpha_val = config.get("alpha", 0.85)
                ah_dict = p_cfg.get("alpha_hierarchy")
                if ah_dict is not None: alpha_val = ah_dict.get(method, alpha_val)
                
                dec_type = p_cfg.get("decimation")
                mask = np.ones(len(d), dtype=bool)
                if dec_type == "interleaved":
                    stride = p_cfg.get("stride", 3)
                    mask = (np.arange(len(d)) % stride) == (m_idx % stride)
                elif dec_type == "density":
                    dy = np.abs(np.gradient(error))
                    mask = (dy > p_cfg.get("density_threshold", 0.005)) | (np.arange(len(d)) % p_cfg.get("sparse_stride", 4) == 0)

                x_err, err_curve = get_plot_curve(d["Distance"].values, error, fit_method, config)
                ax.plot(x_err, err_curve, color=c, linestyle=ls, lw=config["line_width"], alpha=alpha_val)
                
                if show_mk:
                    ax.scatter(d["Distance"].values[mask], error[mask], color=c, marker=mk, 
                               s=ms, edgecolors=config["outline_color"], 
                               linewidths=config["outline_width"], alpha=alpha_val)
        
            ax.set_title(title_map[idx], loc='center')
            ax.set_ylabel(r"Absolute Error (Ha)")
            ax.set_xlabel(r"Distance $d$ ($\AA$)")
            ax.set_yscale("log")
            ax.grid(False) 
            ax.set_xlim(config.get("error_xlim", (0.25, 2.6)))

    ref_panel = "energy_zoomed" if "energy_zoomed" in config["panels"] else "energy_raw"
    leg_cfg = visuals.get(ref_panel, {}) if use_visuals else {}
    
    legend_handles = []
    for method in ["PySCF_FCI"] + methods:
        c = color_map.get(method, "grey")
        
        leg_ls_dict = leg_cfg.get("line_styles")
        leg_ls = leg_ls_dict.get(method, "-" if method != "PySCF_FCI" else "--") if leg_ls_dict else ("-" if method != "PySCF_FCI" else "--")
        
        leg_show_mk = leg_cfg.get("show_markers", True)
        leg_mk = base_marker_map.get(method, "o") if leg_show_mk and method != "PySCF_FCI" else None
        
        clean_label = label_map.get(method, method)
        proxy = Line2D([0], [0], color=c, label=clean_label, marker=leg_mk, markersize=8, linewidth=2, linestyle=leg_ls)
        legend_handles.append(proxy)
        
    fig.legend(handles=legend_handles, loc=config["legend_loc"], bbox_to_anchor=config["legend_bbox"], 
               ncol=len(methods)+1, frameon=True, fancybox=True, edgecolor='black')

    if save_image:
        save_png(save_image, fig)
        save_pdf(save_image, fig)

    plt.show()

In [ ]:
ENABLE_ADVANCED_VISUALS = True

VISUAL_ENHANCEMENTS = {
    "energy_raw": {
        "show_markers": True,
        "line_styles": {"Qiskit_Exact": "-", "Qiskit_VQE": "--"} if False else None,
        "marker_hierarchy": {"Qiskit_Exact": 140, "Qiskit_VQE": 80} if False else None,
        "alpha_hierarchy": {"Qiskit_Exact": 0.5, "Qiskit_VQE": 1.0} if False else None,
        "decimation": "interleaved" if False else None, # interleaved, density ou None
        "stride": 3
    },
    
    "energy_zoomed": {
        "show_markers": True,
        "line_styles": {"Qiskit_Exact": "-", "Qiskit_VQE": "--"} if False else None,
        "marker_hierarchy": {"Qiskit_Exact": 140, "Qiskit_VQE": 80} if False else None,
        "alpha_hierarchy": {"Qiskit_Exact": 0.5, "Qiskit_VQE": 1.0} if False else None,
        "decimation": "interleaved" if False else None,
        "stride": 3
    },
    
    "error_fci": {
        "show_markers": True,
        "line_styles": {"Qiskit_Exact": "-", "Qiskit_VQE": "--"} if False else None,
        "marker_hierarchy": {"Qiskit_Exact": 140, "Qiskit_VQE": 80} if False else None,
        "alpha_hierarchy": {"Qiskit_Exact": 0.5, "Qiskit_VQE": 1.0} if False else None,
        "decimation": "density" if False else None,
        "density_threshold": 0.01,
        "sparse_stride": 4
    }
}

PLOT_CONFIG = {
    "panels": ["energy_raw", "energy_zoomed", "error_fci"],
    
    "fit_methods": {
        "energy_raw": "morse",      #"raw", "spline", "morse", "poly"
        "energy_zoomed": "morse",  
        "error_fci": "raw"       
    },
    
    "poly_degree": 4,             
    "fit_interval": (0.6, 0.9),   
    
    "raw_xlim": (0.25, 2.6),
    "raw_ylim": (-1.2, -0.4),
    "zoom_window": (0.5, 1.0),  
    "zoom_ylim": (-1.18, -1.06), 
    "error_xlim": (0.25, 2.6),    
    
    "line_width": 1.5,
    "marker_size": 50,            
    "alpha": 0.85,                
    "outline_width": 0,           
    "outline_color": "none",      
    "wspace": 0.35,               
    "legend_loc": "upper center",
    "legend_bbox": (0.5, 1.08),
    
    "show_fci_min_line": False,    
    "show_fci_star": False
}

plot_method_comparison(
    df_methods, 
    config=PLOT_CONFIG, 
    visuals=VISUAL_ENHANCEMENTS,
    use_visuals=ENABLE_ADVANCED_VISUALS,
    save_image="h2_method_comparison_final_v3"
)

# 2. 

In [ ]:
distances_dense = np.concatenate([np.linspace(0.3, 0.49, 5), np.linspace(0.5, 1.0, 25), np.linspace(1.1, 2.5, 10)])
distances_dense = np.sort(np.unique(distances_dense))

distances_sparse = np.sort(np.concatenate([np.linspace(0.3, 1.0, 15), np.linspace(1.1, 2.5, 5)]))

records = []

method_data = load_data("h2_method_comparison.pkl")
if method_data and "PySCF_FCI" in method_data:
    fci_list = method_data["PySCF_FCI"]
    for i in range(min(len(fci_list), len(distances_sparse))):
        if fci_list[i] is not None and not np.isnan(fci_list[i]):
            records.append({"Library": "FCI", "Distance": distances_sparse[i], "Energy": fci_list[i]})

files = {
    "PennyLane":     "h2_pennylane_uccsd_results_sampling_dense.pkl",
    "Cirq (Native)": "h2_cirq_uccsd_results_sampling_dense.pkl",
    "Qiskit Nature": "h2_qiskit_uccsd_results_sampling_dense.pkl", 
}

for label, filename in files.items():
    raw_data = load_data(filename)
    if raw_data is not None:
        if isinstance(raw_data, list):
            l = min(len(raw_data), len(distances_dense))
            for i in range(l):
                if raw_data[i] is not None and not np.isnan(raw_data[i]):
                    records.append({"Library": label, "Distance": distances_dense[i], "Energy": raw_data[i]})
        elif isinstance(raw_data, dict):
            s = sorted([(float(k), float(v)) for k, v in raw_data.items() if v is not None])
            for k, v in s:
                records.append({"Library": label, "Distance": k, "Energy": v})

df_libs = pd.DataFrame(records)

In [ ]:
def plot_library_comparison(df, config, visuals, use_visuals=False, save_image=False):
    libraries = [lib for lib in df["Library"].unique() if lib != "FCI"]
    
    color_map = {
        "Qiskit Nature": "#0072B2", 
        "PennyLane": "#D55E00",     
        "Cirq (Native)": "#009E73",          
        "FCI": "#000000"            
    }
    
    base_marker_map = {
        "Qiskit Nature": "o",
        "PennyLane": "s",
        "Cirq (Native)": "^",
        "FCI": ""  
    }

    fig, axes = plt.subplots(1, len(config["panels"]), figsize=(18, 5))
    if len(config["panels"]) == 1: axes = [axes]
    plt.subplots_adjust(wspace=config["wspace"])
    
    fci_data_df = df[df["Library"] == "FCI"].sort_values("Distance")
    fci_dist = fci_data_df["Distance"].values
    fci_energies = fci_data_df["Energy"].values

    def morse_func(r, De, a, re, E_shift):
        return De * (1 - np.exp(-a * (r - re)))**2 + E_shift

    def get_plot_curve(x, y, fit_type, cfg):
        idx = np.argsort(x)
        x, y = x[idx], y[idx]
        
        if fit_type == "raw": return x, y
        
        x_dense = np.linspace(min(x), max(x), 500)
        
        if fit_type == "spline":
            try: return x_dense, make_interp_spline(x, y, k=3)(x_dense)
            except: return x, y
            
        elif fit_type == "morse":
            try:
                mask = x > 0.45
                popt, _ = curve_fit(morse_func, x[mask], y[mask], p0=[0.2, 1.5, 0.74, -1.137], maxfev=20000)
                return x_dense, morse_func(x_dense, *popt)
            except: return x, y
            
        elif fit_type == "poly":
            interval = cfg.get("fit_interval", (0.5, 1.0))
            mask = (x >= interval[0]) & (x <= interval[1])
            if len(x[mask]) > cfg.get("poly_degree", 4):
                coeffs = np.polyfit(x[mask], y[mask], cfg.get("poly_degree", 4))
                x_local = np.linspace(interval[0], interval[1], 300)
                return x_local, np.poly1d(coeffs)(x_local)
            return x, y
            
        return x, y 

    fci_min_x, fci_min_y = None, None
    if len(fci_dist) > 0:
        zoom_fit = config.get("fit_methods", {}).get("energy_zoomed", "spline")
        f_x, f_y = get_plot_curve(fci_dist, fci_energies, zoom_fit, config)
        f_min_idx = np.argmin(f_y)
        fci_min_x, fci_min_y = f_x[f_min_idx], f_y[f_min_idx]

    title_map = {0: "(a)", 1: "(b)", 2: "(c)"}

    for idx, panel in enumerate(config["panels"]):
        ax = axes[idx]
        p_cfg = visuals.get(panel, {}) if use_visuals else {}
        
        fit_method = config.get("fit_methods", {}).get(panel, "raw")
        
        if panel == "energy_raw" or panel == "energy_zoomed":
            for lib_idx, lib in enumerate(["FCI"] + libraries):
                d = df[df["Library"] == lib].sort_values("Distance")
                if len(d) == 0: continue
                
                c = color_map.get(lib, "grey")
                mk = base_marker_map.get(lib, "o")
                
                ls_dict = p_cfg.get("line_styles")
                ls = ls_dict.get(lib, "-" if lib != "FCI" else "--") if ls_dict else ("-" if lib != "FCI" else "--")
                show_mk = p_cfg.get("show_markers", True) and lib != "FCI"
                
                ms = config.get("marker_size", 50)
                mh_dict = p_cfg.get("marker_hierarchy")
                if mh_dict is not None: ms = mh_dict.get(lib, ms)
                if panel == "energy_zoomed": ms *= 1.5
                
                alpha_val = config.get("alpha", 0.85)
                ah_dict = p_cfg.get("alpha_hierarchy")
                if ah_dict is not None and lib != "FCI": alpha_val = ah_dict.get(lib, alpha_val)
                if lib == "FCI": alpha_val = config.get("alpha", 0.85)
                
                x_raw, y_raw = d["Distance"].values, d["Energy"].values
                x_curve, y_curve = get_plot_curve(x_raw, y_raw, fit_method, config)
                
                dec_type = p_cfg.get("decimation")
                mask = np.ones(len(x_raw), dtype=bool)
                if dec_type == "interleaved" and lib != "FCI":
                    stride = p_cfg.get("stride", 3)
                    mask = (np.arange(len(x_raw)) % stride) == (lib_idx % stride)
                elif dec_type == "density" and lib != "FCI":
                    dy = np.abs(np.gradient(y_raw))
                    mask = (dy > p_cfg.get("density_threshold", 0.05)) | (np.arange(len(x_raw)) % p_cfg.get("sparse_stride", 4) == 0)

                lw = 2.5 if lib == "FCI" else config["line_width"] 
                ax.plot(x_curve, y_curve, color=c, linestyle=ls, lw=lw, alpha=alpha_val, zorder=2)
                
                if show_mk: 
                    ax.scatter(x_raw[mask], y_raw[mask], color=c, marker=mk, 
                               s=ms, edgecolors=config["outline_color"], 
                               linewidths=config["outline_width"], zorder=3, alpha=alpha_val)
                
            if panel == "energy_zoomed" and fci_min_x is not None:
                if config.get("show_fci_min_line", False):
                    ax.axvline(x=fci_min_x, color='black', linestyle=':', alpha=0.5, zorder=1)
                if config.get("show_fci_star", False):
                    ax.scatter(fci_min_x, fci_min_y, color='red', marker='*', 
                               s=config.get("marker_size", 50) * 6, edgecolors='none', zorder=10)

            ax.set_title(title_map[idx], loc='center')
            ax.set_ylabel("Energy (Ha)")
            ax.set_xlabel(r"Distance $d$ ($\AA$)")
            ax.grid(False)
            
            if panel == "energy_raw":
                ax.set_xlim(config.get("raw_xlim", (0.25, 2.6)))
                ax.set_ylim(config.get("raw_ylim", (-1.2, -0.4)))
            elif panel == "energy_zoomed":
                ax.set_xlim(config.get("zoom_window", (0.55, 0.95)))
                ax.set_ylim(config.get("zoom_ylim", (-1.145, -1.105)))
                
        elif panel == "error_fci":
            for lib_idx, lib in enumerate(libraries):
                d = df[df["Library"] == lib].sort_values("Distance")
                if len(d) == 0: continue
                
                fci_interp = np.interp(d["Distance"].values, fci_dist, fci_energies)
                error = np.abs(d["Energy"].values - fci_interp)
                
                c = color_map.get(lib, "grey")
                mk = base_marker_map.get(lib, "o")
                
                ls_dict = p_cfg.get("line_styles")
                ls = ls_dict.get(lib, "-") if ls_dict else "-"
                
                show_mk = p_cfg.get("show_markers", True)
                ms = config.get("marker_size", 50)
                mh_dict = p_cfg.get("marker_hierarchy")
                if mh_dict is not None: ms = mh_dict.get(lib, ms)
                
                alpha_val = config.get("alpha", 0.85)
                ah_dict = p_cfg.get("alpha_hierarchy")
                if ah_dict is not None: alpha_val = ah_dict.get(lib, alpha_val)
                
                dec_type = p_cfg.get("decimation")
                mask = np.ones(len(d), dtype=bool)
                if dec_type == "interleaved":
                    stride = p_cfg.get("stride", 3)
                    mask = (np.arange(len(d)) % stride) == (lib_idx % stride)
                elif dec_type == "density":
                    dy = np.abs(np.gradient(error))
                    mask = (dy > p_cfg.get("density_threshold", 0.005)) | (np.arange(len(d)) % p_cfg.get("sparse_stride", 4) == 0)

                x_err, err_curve = get_plot_curve(d["Distance"].values, error, fit_method, config)
                ax.plot(x_err, err_curve, color=c, linestyle=ls, lw=config["line_width"], alpha=alpha_val)
                
                if show_mk:
                    ax.scatter(d["Distance"].values[mask], error[mask], color=c, marker=mk, 
                               s=ms, edgecolors=config["outline_color"], 
                               linewidths=config["outline_width"], alpha=alpha_val)
        
            ax.set_title(title_map[idx], loc='center')
            ax.set_ylabel(r"Absolute Error (Ha)")
            ax.set_xlabel(r"Distance $d$ ($\AA$)")
            ax.set_yscale("log")
            ax.grid(False) 
            ax.set_xlim(config.get("error_xlim", (0.25, 2.6)))

    ref_panel = "energy_zoomed" if "energy_zoomed" in config["panels"] else "energy_raw"
    leg_cfg = visuals.get(ref_panel, {}) if use_visuals else {}
    
    legend_handles = []
    for lib in ["FCI"] + libraries:
        c = color_map.get(lib, "grey")
        
        leg_ls_dict = leg_cfg.get("line_styles")
        leg_ls = leg_ls_dict.get(lib, "-" if lib != "FCI" else "--") if leg_ls_dict else ("-" if lib != "FCI" else "--")
        
        leg_show_mk = leg_cfg.get("show_markers", True)
        leg_mk = base_marker_map.get(lib, "o") if leg_show_mk and lib != "FCI" else None
        
        proxy = Line2D([0], [0], color=c, label=lib, marker=leg_mk, markersize=8, linewidth=2, linestyle=leg_ls)
        legend_handles.append(proxy)
        
    fig.legend(handles=legend_handles, loc=config["legend_loc"], bbox_to_anchor=config["legend_bbox"], 
               ncol=len(libraries)+1, frameon=True, fancybox=True, edgecolor='black')

    if save_image:
        save_png(save_image, fig)
        save_pdf(save_image, fig)

    plt.show()

In [ ]:
ENABLE_ADVANCED_VISUALS = True

VISUAL_ENHANCEMENTS = {
    "energy_raw": {
        "show_markers": True if True else False,
        "line_styles": {"Qiskit Nature": "-", "PennyLane": "--", "Cirq (Native)": ":"} if False else None,
        "marker_hierarchy": {"Qiskit Nature": 180, "PennyLane": 100, "Cirq (Native)": 50} if False else None,
        #"marker_hierarchy": {"Cirq (Native)": 30, "PennyLane": 80, "Qiskit Nature": 180} if True else None,
        "alpha_hierarchy": {"Qiskit Nature": 0.5, "PennyLane": 0.75, "Cirq (Native)": 1.0} if False else None,
        "decimation": "interleaved" if True else None,
        "stride": 3,
        "density_threshold": 0.01,
        "sparse_stride": 4
    },
        
    "energy_zoomed": {
        "show_markers": True if True else False,
        "line_styles": {"Qiskit Nature": "-", "PennyLane": "--", "Cirq (Native)": ":"} if False else None,
        "marker_hierarchy": {"Qiskit Nature": 140, "PennyLane": 80, "Cirq (Native)": 30} if False else None,
        "alpha_hierarchy": {"Qiskit Nature": 0.5, "PennyLane": 0.75, "Cirq (Native)": 1.0} if False else None,
        "decimation": "interleaved" if False else None,
        "stride": 3
    },
    
    "error_fci": {
        "show_markers": True if True else False,
        "line_styles": {"Qiskit Nature": "-", "PennyLane": "--", "Cirq (Native)": ":"} if False else None,
        "marker_hierarchy": {"Qiskit Nature": 140, "PennyLane": 80, "Cirq (Native)": 30} if False else None,
        "alpha_hierarchy": {"Qiskit Nature": 0.5, "PennyLane": 0.75, "Cirq (Native)": 1.0} if False else None,
        "decimation": "density" if False else None,
        "density_threshold": 0.01,
        "sparse_stride": 4
    }
}

PLOT_CONFIG_LIBS = {
    "panels": ["energy_raw", "energy_zoomed"],
    
    "fit_methods": {
        "energy_raw": "morse",      
        "energy_zoomed": "morse",  
        "error_fci": "raw"        
    },
    
    "poly_degree": 4,             
    "fit_interval": (0.6, 0.9),   
    
    "raw_xlim": (0.25, 2.6),
    "raw_ylim": (-1.2, -0.4),
    "zoom_window": (0.55, 0.95),  
    "zoom_ylim": (-1.145, -1.100), 
    "error_xlim": (0.25, 2.6),    
    
    "line_width": 1.5,
    "marker_size": 60,            
    "alpha": 0.85,                
    "outline_width": 0,           
    "outline_color": "none",      
    "wspace": 0.35,               
    "legend_loc": "upper center",
    "legend_bbox": (0.5, 1.08),
    
    "show_fci_min_line": True,    
    "show_fci_star": True
}

plot_library_comparison(
    df_libs, 
    config=PLOT_CONFIG_LIBS, 
    visuals=VISUAL_ENHANCEMENTS,
    use_visuals=ENABLE_ADVANCED_VISUALS,
    save_image="h2_library_comparison_final_v3"
)

# 3. 

In [ ]:
distances_opt = np.sort(np.concatenate([np.linspace(0.3, 1.0, 10), np.linspace(1.1, 2.5, 8)]))
distances_fci = np.sort(np.concatenate([np.linspace(0.3, 1.0, 15), np.linspace(1.1, 2.5, 5)]))

records = []

method_data = load_data("h2_method_comparison.pkl")
if method_data and "PySCF_FCI" in method_data:
    fci_list = method_data["PySCF_FCI"]
    for i in range(min(len(fci_list), len(distances_fci))):
        if fci_list[i] is not None and not np.isnan(fci_list[i]):
            for m in ["aer_exact", "aer_sampling"]:
                records.append({
                    "Method": m, "Optimizer": "FCI", "Distance": distances_fci[i], 
                    "Energy": fci_list[i], "Iterations": np.nan, "Evaluations": np.nan
                })

raw_data = load_data("h2_optimiser_comparison_4096_nomaxiter_V2.pkl")
excluded_optimizers = ["L-BFGS-B", "TNC"]

if raw_data:
    for method, opts_dict in raw_data.items():
        for opt_name, data_list in opts_dict.items():
            if opt_name in excluded_optimizers: continue
            for i, item in enumerate(data_list):
                if item is None: continue
                dist = distances_opt[i] if i < len(distances_opt) else distances_opt[-1]
                
                energy, elapsed, metrics = item
                records.append({
                    "Method": method, 
                    "Optimizer": opt_name, 
                    "Distance": dist,
                    "Energy": energy, 
                    "Evaluations": metrics.get('cost_function_evals', metrics.get('nfev', np.nan)),
                    "Iterations": metrics.get('nit', np.nan)
                })

df_opts = pd.DataFrame(records)

In [ ]:
import matplotlib.ticker as ticker

def plot_optimizer_comparison_dual(df, config, visuals, use_visuals=False, save_image=False):
    methods = config.get("methods", ["aer_exact", "aer_sampling"])
    
    subset = df[df["Method"].isin(methods)]
    if subset.empty: return
    optimizers = [opt for opt in subset["Optimizer"].unique() if opt != "FCI"]
    
    palette = sns.color_palette("tab10", n_colors=len(optimizers))
    color_map = dict(zip(optimizers, palette))
    color_map["FCI"] = "#000000"
    
    base_marker_map = dict(zip(optimizers, config["markers"][:len(optimizers)]))
    base_marker_map["FCI"] = ""

    nrows = len(methods)
    panels_grid = [config.get("panels_row1", []), config.get("panels_row2", [])]
    ncols = max(len(panels_grid[0]), len(panels_grid[1]))
    
    fig_w = config.get("fig_width", 22)
    fig_h = config.get("fig_height", 11)
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h))
    if ncols == 1: axes = axes.reshape(nrows, 1)
        
    plt.subplots_adjust(wspace=config["wspace"], hspace=config.get("hspace", 0.35))

    def morse_func(r, De, a, re, E_shift):
        return De * (1 - np.exp(-a * (r - re)))**2 + E_shift

    def get_plot_curve(x, y, fit_type, cfg):
        idx = np.argsort(x)
        x, y = x[idx], y[idx]
        
        if fit_type == "raw": return x, y
        
        x_dense = np.linspace(min(x), max(x), 500)
        
        if fit_type == "spline":
            try: 
                x_clean, unique_idx = np.unique(x, return_index=True)
                y_clean = y[unique_idx]
                if len(x_clean) > 3: return x_dense, make_interp_spline(x_clean, y_clean, k=3)(x_dense)
                return x, y
            except: return x, y
            
        elif fit_type == "morse":
            try:
                mask = x > 0.45
                popt, _ = curve_fit(morse_func, x[mask], y[mask], p0=[0.2, 1.5, 0.74, -1.137], maxfev=20000)
                return x_dense, morse_func(x_dense, *popt)
            except: return x, y
            
        elif fit_type == "poly":
            interval = cfg.get("fit_interval", (0.5, 1.0))
            mask = (x >= interval[0]) & (x <= interval[1])
            if len(x[mask]) > cfg.get("poly_degree", 4):
                coeffs = np.polyfit(x[mask], y[mask], cfg.get("poly_degree", 4))
                x_local = np.linspace(interval[0], interval[1], 300)
                return x_local, np.poly1d(coeffs)(x_local)
            return x, y
            
        return x, y 

    letter_idx = 0
    legend_built = False
    legend_handles = []

    for row_idx, method in enumerate(methods):
        method_suffix = method.split('_')[1] 
        panels = panels_grid[row_idx]
        
        for col_idx in range(len(panels), ncols):
            axes[row_idx, col_idx].set_visible(False)
            
        fci_data_df = df[(df["Method"] == method) & (df["Optimizer"] == "FCI")].sort_values("Distance")
        fci_dist = fci_data_df["Distance"].values
        fci_energies = fci_data_df["Energy"].values

        fci_min_x, fci_min_y = None, None
        if len(fci_dist) > 0:
            zoom_panel_id = f"energy_zoomed_{method_suffix}"
            zoom_fit = config.get("fit_methods", {}).get(zoom_panel_id, "morse")
            f_x, f_y = get_plot_curve(fci_dist, fci_energies, zoom_fit, config)
            f_min_idx = np.argmin(f_y)
            fci_min_x, fci_min_y = f_x[f_min_idx], f_y[f_min_idx]

        for col_idx, panel in enumerate(panels):
            ax = axes[row_idx, col_idx]
            
            panel_id = f"{panel}_{method_suffix}"
            p_cfg = visuals.get(panel_id, {}) if use_visuals else {}
            fit_method = config.get("fit_methods", {}).get(panel_id, "raw")
            
            y_col = "Energy"
            if panel == "iterations": y_col = "Iterations"
            elif panel == "evaluations": y_col = "Evaluations"
            
            opts_to_plot = ["FCI"] + optimizers if panel in ["energy_raw", "energy_zoomed"] else optimizers
            
            x_jitter = p_cfg.get("x_jitter", 0.0)
            groups = p_cfg.get("groups", [])
            if not groups: 
                groups = [optimizers] 
            
            for opt_idx, opt in enumerate(opts_to_plot):
                d = df[(df["Method"] == method) & (df["Optimizer"] == opt)].sort_values("Distance").dropna(subset=[y_col])
                if len(d) == 0: continue
                
                c = color_map.get(opt, "grey")
                mk = base_marker_map.get(opt, "o")
                
                ls_dict = p_cfg.get("line_styles")
                ls = ls_dict.get(opt, "-" if opt != "FCI" else "--") if ls_dict else ("-" if opt != "FCI" else "--")
                show_mk = p_cfg.get("show_markers", True) and opt != "FCI"
                
                ms = config.get("marker_size", 50)
                mh_dict = p_cfg.get("marker_hierarchy")
                if mh_dict is not None: ms = mh_dict.get(opt, ms)
                if panel == "energy_zoomed": ms *= 1.5
                
                alpha_val = config.get("alpha", 0.85)
                ah_dict = p_cfg.get("alpha_hierarchy")
                if ah_dict is not None and opt != "FCI": alpha_val = ah_dict.get(opt, alpha_val)
                if opt == "FCI": alpha_val = config.get("alpha", 0.85)
                
                x_raw, y_raw = d["Distance"].values, d[y_col].values
                x_curve, y_curve = get_plot_curve(x_raw, y_raw, fit_method, config)
                
                dec_type = p_cfg.get("decimation")
                mask = np.ones(len(x_raw), dtype=bool)
                shift = 0.0
                
                if opt != "FCI":
                    
                    my_group = next((g for g in groups if opt in g), [opt])
                    group_size = len(my_group)
                    idx_in_group = my_group.index(opt)
                    
                    if x_jitter > 0 and group_size > 1:
                        
                        shift = (idx_in_group - (group_size - 1) / 2.0) * x_jitter
                    
                    if dec_type == "interleaved":
                        
                        stride = p_cfg.get("stride", 3)
                        mask = (np.arange(len(x_raw)) % stride) == (idx_in_group % stride)
                    elif dec_type == "density":
                        
                        dy = np.abs(np.gradient(y_raw))
                        thresh = p_cfg.get("density_threshold", 0.05)
                        sparse_stride = p_cfg.get("sparse_stride", 4)
                        mask = (dy > thresh) | ((np.arange(len(x_raw)) - idx_in_group) % sparse_stride == 0)

                lw = 2.5 if opt == "FCI" else config["line_width"] 

                ax.plot(x_curve, y_curve, color=c, linestyle=ls, lw=lw, alpha=alpha_val, zorder=2)
                
                if show_mk: 

                    ax.scatter(x_raw[mask] + shift, y_raw[mask], color=c, marker=mk, 
                               s=ms, edgecolors=config["outline_color"], 
                               linewidths=config["outline_width"], zorder=3, alpha=alpha_val)
                    
                if not legend_built and row_idx == 0 and col_idx == 0:
                    proxy = Line2D([0], [0], color=c, label=opt, marker=mk if show_mk and opt != "FCI" else None, 
                                   markersize=8, linewidth=2, linestyle=ls)
                    legend_handles.append(proxy)
                    
            if panel == "energy_zoomed" and fci_min_x is not None:
                if config.get("show_fci_min_line", False):
                    ax.axvline(x=fci_min_x, color='black', linestyle=':', alpha=0.5, zorder=1)
                if config.get("show_fci_star", False):
                    ax.scatter(fci_min_x, fci_min_y, color='red', marker='*', 
                               s=config.get("marker_size", 50) * 6, edgecolors='none', zorder=10)

            ax.set_title(f"({chr(97 + letter_idx)})", loc='center')
            letter_idx += 1
            
            ax.set_xlabel(r"Distance $d$ ($\AA$)")
            ax.grid(False)
            
            if panel == "energy_raw":
              
                ax.set_ylabel("Energy (Ha)")
                ax.set_xlim(config.get(f"xlim_energy_raw_{method_suffix}", (0.25, 2.6)))
                ax.set_ylim(config.get(f"ylim_energy_raw_{method_suffix}", (-1.2, -0.4)))
                
            elif panel == "energy_zoomed":
               
                ax.set_ylabel("Energy (Ha)")
                ax.set_xlim(config.get(f"xlim_energy_zoomed_{method_suffix}", (0.55, 0.95)))
                ax.set_ylim(config.get(f"ylim_energy_zoomed_{method_suffix}", (-1.145, -1.100)))
           
            elif panel == "iterations":
                
                ax.set_ylabel("Iterations (nit)")
                ax.set_xlim(config.get(f"xlim_energy_raw_{method_suffix}", (0.25, 2.6)))
           
            elif panel == "evaluations":
                
                ax.set_ylabel("Circuit Executions (nfev)")
                ax.set_xlim(config.get(f"xlim_energy_raw_{method_suffix}", (0.25, 2.6)))

            y_decimals = config.get("y_decimals", {}).get(panel_id)
          
            if y_decimals is not None:
                ax.yaxis.set_major_formatter(ticker.FormatStrFormatter(f'%.{y_decimals}f'))
                
            x_decimals = config.get("x_decimals", {}).get(panel_id)
          
            if x_decimals is not None:
                ax.xaxis.set_major_formatter(ticker.FormatStrFormatter(f'%.{x_decimals}f'))

        legend_built = True 

    fig.legend(handles=legend_handles, loc=config["legend_loc"], bbox_to_anchor=config["legend_bbox"], 
               ncol=len(optimizers)+1, frameon=True, fancybox=True, edgecolor='black')

    if save_image:
        save_png(save_image, fig)
        save_pdf(save_image, fig)

    plt.show()

In [ ]:
ENABLE_ADVANCED_VISUALS = True

VISUAL_ENHANCEMENTS = {
    "energy_raw_exact": {
        "show_markers": True,
        "line_styles": {"COBYLA": "-", "SPSA": "--", "SLSQP": ":"} if False else None,
        "marker_hierarchy": {"COBYLA": 140, "SPSA": 80, "SLSQP": 30} if False else None,
        "alpha_hierarchy": {"COBYLA": 0.5, "SPSA": 0.75, "SLSQP": 1.0} if False else None,
        
        "x_jitter": 0.0,
        "groups": [],
        
        "decimation": "interleaved", 
        "stride": 1, "density_threshold": 0.05, "sparse_stride": 4
    },
    "energy_zoomed_exact": {
        "show_markers": True,
        "line_styles": {"COBYLA": "-", "SPSA": "--", "SLSQP": ":"} if False else None,
        "marker_hierarchy": {"COBYLA": 140, "SPSA": 80, "SLSQP": 30} if False else None,
        "alpha_hierarchy": {"COBYLA": 0.5, "SPSA": 0.75, "SLSQP": 1.0} if False else None,
        
        "x_jitter": 0.0,
        "groups": [],
        
        "decimation": None,
        "stride": 4, "density_threshold": 0.05, "sparse_stride": 4
    },
    "iterations_exact": {
        "show_markers": True,
        "line_styles": {"COBYLA": "-", "SPSA": "--", "SLSQP": ":"} if False else None,
        "marker_hierarchy": {"COBYLA": 140, "SPSA": 80, "SLSQP": 30} if False else None,
        "alpha_hierarchy": {"COBYLA": 0.5, "SPSA": 0.75, "SLSQP": 1.0} if False else None,
        "x_jitter": 0.0, "groups": [],
        "decimation": None, 
        "stride": 4, "density_threshold": 10.0, "sparse_stride": 4
    },
    "evaluations_exact": {
        "show_markers": True,
        "line_styles": {"COBYLA": "-", "SPSA": "--", "SLSQP": ":"} if False else None,
        "marker_hierarchy": {"COBYLA": 140, "SPSA": 80, "SLSQP": 30} if False else None,
        "alpha_hierarchy": {"COBYLA": 0.5, "SPSA": 0.75, "SLSQP": 1.0} if False else None,
        "x_jitter": 0.0, "groups": [],
        "decimation": None,
        "stride": 4, "density_threshold": 50.0, "sparse_stride": 4
    },
    
    "energy_raw_sampling": {
        "show_markers": True,
        "line_styles": {"COBYLA": "-", "SPSA": "--", "SLSQP": ":"} if False else None,
        "marker_hierarchy": {"COBYLA": 140, "SPSA": 80, "SLSQP": 30} if False else None,
        "alpha_hierarchy": {"COBYLA": 0.5, "SPSA": 0.75, "SLSQP": 1.0} if False else None,
        "x_jitter": 0.0, "groups": [],
        "decimation": "interleaved",
        "stride": 1, "density_threshold": 0.05, "sparse_stride": 4
    },
    "energy_zoomed_sampling": {
        "show_markers": True,
        "line_styles": {"COBYLA": "-", "SPSA": "--", "SLSQP": ":"} if False else None,
        "marker_hierarchy": {"COBYLA": 140, "SPSA": 80, "SLSQP": 30} if False else None,
        "alpha_hierarchy": {"COBYLA": 0.5, "SPSA": 0.75, "SLSQP": 1.0} if False else None,
        "x_jitter": 0.0, "groups": [],
        "decimation": None,
        "stride": 4, "density_threshold": 0.05, "sparse_stride": 4
    },
    "iterations_sampling": {
        "show_markers": True,
        "line_styles": {"COBYLA": "-", "SPSA": "--", "SLSQP": ":"} if False else None,
        "marker_hierarchy": {"COBYLA": 140, "SPSA": 80, "SLSQP": 30} if False else None,
        "alpha_hierarchy": {"COBYLA": 0.5, "SPSA": 0.75, "SLSQP": 1.0} if False else None,
        "x_jitter": 0.0, "groups": [],
        "decimation": None,
        "stride": 4, "density_threshold": 10.0, "sparse_stride": 4
    },
    "evaluations_sampling": {
        "show_markers": True,
        "line_styles": {"COBYLA": "-", "SPSA": "--", "SLSQP": ":"} if False else None,
        "marker_hierarchy": {"COBYLA": 140, "SPSA": 80, "SLSQP": 30} if False else None,
        "alpha_hierarchy": {"COBYLA": 0.5, "SPSA": 0.75, "SLSQP": 1.0} if False else None,
        "x_jitter": 0.0, "groups": [],
        "decimation": None,
        "stride": 4, "density_threshold": 50.0, "sparse_stride": 4
    }
}

PLOT_CONFIG_DUAL = {
    "methods": ["aer_exact", "aer_sampling"], 
    
    #"panels_row1": ["energy_raw", "energy_zoomed", "iterations", "evaluations"],
    "panels_row1": ["energy_raw", "energy_zoomed"],
    #"panels_row2": ["energy_raw", "energy_zoomed", "iterations", "evaluations"],
    "panels_row2": ["energy_raw", "energy_zoomed"],
    
    "fit_methods": {
        "energy_raw_exact": "morse",      
        "energy_zoomed_exact": "morse",  
        "iterations_exact": "raw",      
        "evaluations_exact": "raw",
        "energy_raw_sampling": "morse",      
        "energy_zoomed_sampling": "morse",  
        "iterations_sampling": "raw",      
        "evaluations_sampling": "raw"
    },
    
    "y_decimals": {
        "energy_zoomed_exact": 3,
        "energy_zoomed_sampling": 3,
        "energy_raw_exact": 2,
        "energy_raw_sampling": 2
    },
    
    "poly_degree": 4,             
    "fit_interval": (0.6, 0.9),   
    
    "xlim_energy_raw_exact": (0.25, 2.6),
    "ylim_energy_raw_exact": (-1.2, -0.4),
    "xlim_energy_zoomed_exact": (0.55, 0.95),
    "ylim_energy_zoomed_exact": (-1.145, -1.100),
    
    "xlim_energy_raw_sampling": (0.25, 2.6),
    "ylim_energy_raw_sampling": (-1.2, 0.1),
    "xlim_energy_zoomed_sampling": (0.55, 0.95),
    "ylim_energy_zoomed_sampling": (-1.145, -1.000),
    
    "fig_width": 22,
    "fig_height": 11,
    
    "line_width": 1.5,
    "marker_size": 50,            
    "alpha": 0.85,                
    "outline_width": 0,           
    "outline_color": "none",      
    
    "wspace": 0.35,  
    "hspace": 0.35,  
    
    "legend_loc": "upper center",
    "legend_bbox": (0.5, 0.99),  
    "markers": ['o', 's', '^', 'D', 'X', 'P', 'v', '*'],
    
    "show_fci_min_line": True,    
    "show_fci_star": True
}

plot_optimizer_comparison_dual(
    df_opts, 
    config=PLOT_CONFIG_DUAL, 
    visuals=VISUAL_ENHANCEMENTS,
    use_visuals=ENABLE_ADVANCED_VISUALS,
    #save_image="h2_optimizer_dual_level_final"
)

# 3.5

In [ ]:
distances_opt = np.sort(np.concatenate([np.linspace(0.3, 1.0, 10), np.linspace(1.1, 2.5, 8)]))
distances_fci = np.sort(np.concatenate([np.linspace(0.3, 1.0, 15), np.linspace(1.1, 2.5, 5)]))

records = []

method_data = load_data("h2_method_comparison.pkl")
if method_data and "PySCF_FCI" in method_data:
    fci_list = method_data["PySCF_FCI"]
    for i in range(min(len(fci_list), len(distances_fci))):
        if fci_list[i] is not None and not np.isnan(fci_list[i]):
            for m in ["aer_exact", "aer_sampling"]:
                records.append({
                    "Method": m, "Optimizer": "FCI", "Distance": distances_fci[i], 
                    "Energy": fci_list[i], "Iterations": np.nan, "Evaluations": np.nan
                })

raw_data = load_data("h2_optimiser_comparison_4096_nomaxiter_V2.pkl")
excluded_optimizers = ["L-BFGS-B", "TNC"]

if raw_data:
    for method, opts_dict in raw_data.items():
        for opt_name, data_list in opts_dict.items():
            if opt_name in excluded_optimizers: continue
            for i, item in enumerate(data_list):
                if item is None: continue
                dist = distances_opt[i] if i < len(distances_opt) else distances_opt[-1]
                
                energy, elapsed, metrics = item
                records.append({
                    "Method": method, 
                    "Optimizer": opt_name, 
                    "Distance": dist,
                    "Energy": energy, 
                    "Evaluations": metrics.get('cost_function_evals', metrics.get('nfev', np.nan)),
                    "Iterations": metrics.get('nit', np.nan)
                })

df_opts = pd.DataFrame(records)

In [ ]:
import matplotlib.ticker as ticker

def plot_optimizer_comparison_dual(df, config, visuals, use_visuals=False, save_image=False):
    methods = config.get("methods", ["aer_exact", "aer_sampling"])
    
    subset = df[df["Method"].isin(methods)]
    if subset.empty: return
    optimizers = [opt for opt in subset["Optimizer"].unique() if opt != "FCI"]
    
    palette = sns.color_palette("tab10", n_colors=len(optimizers))
    color_map = dict(zip(optimizers, palette))
    color_map["FCI"] = "#000000"
    
    base_marker_map = dict(zip(optimizers, config["markers"][:len(optimizers)]))
    base_marker_map["FCI"] = ""

    nrows = len(methods)
    panels_grid = [config.get("panels_row1", []), config.get("panels_row2", [])]
    ncols = max(len(panels_grid[0]), len(panels_grid[1]))
    
    fig_w = config.get("fig_width", 18)
    fig_h = config.get("fig_height", 10)
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h))
    if ncols == 1: axes = axes.reshape(nrows, 1)
        
    plt.subplots_adjust(wspace=config["wspace"], hspace=config.get("hspace", 0.35))

    def morse_func(r, De, a, re, E_shift):
        return De * (1 - np.exp(-a * (r - re)))**2 + E_shift

    def get_plot_curve(x, y, fit_type, cfg):
        idx = np.argsort(x)
        x, y = x[idx], y[idx]
        
        if fit_type == "raw": return x, y
        
        x_dense = np.linspace(min(x), max(x), 500)
        
        if fit_type == "spline":
            try: 
                x_clean, unique_idx = np.unique(x, return_index=True)
                y_clean = y[unique_idx]
                if len(x_clean) > 3: return x_dense, make_interp_spline(x_clean, y_clean, k=3)(x_dense)
                return x, y
            except: return x, y
            
        elif fit_type == "morse":
            try:
                mask = x > 0.45
                popt, _ = curve_fit(morse_func, x[mask], y[mask], p0=[0.2, 1.5, 0.74, -1.137], maxfev=20000)
                return x_dense, morse_func(x_dense, *popt)
            except: return x, y
            
        elif fit_type == "poly":
            interval = cfg.get("fit_interval", (0.5, 1.0))
            mask = (x >= interval[0]) & (x <= interval[1])
            if len(x[mask]) > cfg.get("poly_degree", 4):
                coeffs = np.polyfit(x[mask], y[mask], cfg.get("poly_degree", 4))
                x_local = np.linspace(interval[0], interval[1], 300)
                return x_local, np.poly1d(coeffs)(x_local)
            return x, y
            
        return x, y 

    def get_param(param_val, g_idx):
        if isinstance(param_val, list):
            return param_val[g_idx] if g_idx < len(param_val) else param_val[-1]
        return param_val

    letter_idx = 0
    legend_built = False
    legend_handles = []

    for row_idx, method in enumerate(methods):
        method_suffix = method.split('_')[1] 
        panels = panels_grid[row_idx]
        
        for col_idx in range(len(panels), ncols):
            axes[row_idx, col_idx].set_visible(False)
            
        fci_data_df = df[(df["Method"] == method) & (df["Optimizer"] == "FCI")].sort_values("Distance")
        fci_dist = fci_data_df["Distance"].values
        fci_energies = fci_data_df["Energy"].values

        fci_min_x, fci_min_y = None, None
        if len(fci_dist) > 0:
            zoom_panel_id = f"energy_zoomed_{method_suffix}"
            zoom_fit = config.get("fit_methods", {}).get(zoom_panel_id, "morse")
            f_x, f_y = get_plot_curve(fci_dist, fci_energies, zoom_fit, config)
            f_min_idx = np.argmin(f_y)
            fci_min_x, fci_min_y = f_x[f_min_idx], f_y[f_min_idx]

        for col_idx, panel in enumerate(panels):
            ax = axes[row_idx, col_idx]
            
            panel_id = f"{panel}_{method_suffix}"
            p_cfg = visuals.get(panel_id, {}) if use_visuals else {}
            fit_method = config.get("fit_methods", {}).get(panel_id, "raw")
            
            y_col = "Energy"
            if panel == "iterations": y_col = "Iterations"
            elif panel == "evaluations": y_col = "Evaluations"
            
            opts_to_plot = ["FCI"] + optimizers if panel in ["energy_raw", "energy_zoomed"] else optimizers
            
            groups = p_cfg.get("groups", [])
            if not groups: 
                groups = [optimizers] 
            
            for opt_idx, opt in enumerate(opts_to_plot):
                d = df[(df["Method"] == method) & (df["Optimizer"] == opt)].sort_values("Distance").dropna(subset=[y_col])
                if len(d) == 0: continue
                
                c = color_map.get(opt, "grey")
                mk = base_marker_map.get(opt, "o")
                
                ls_dict = p_cfg.get("line_styles")
                ls = ls_dict.get(opt, "-" if opt != "FCI" else "--") if ls_dict else ("-" if opt != "FCI" else "--")
                show_mk = p_cfg.get("show_markers", True) and opt != "FCI"
                
                ms = config.get("marker_size", 50)
                mh_dict = p_cfg.get("marker_hierarchy")
                if mh_dict is not None: ms = mh_dict.get(opt, ms)
                if panel == "energy_zoomed": ms *= 1.5
                
                alpha_val = config.get("alpha", 0.85)
                ah_dict = p_cfg.get("alpha_hierarchy")
                if ah_dict is not None and opt != "FCI": alpha_val = ah_dict.get(opt, alpha_val)
                if opt == "FCI": alpha_val = config.get("alpha", 0.85)
                
                x_raw, y_raw = d["Distance"].values, d[y_col].values
                x_curve, y_curve = get_plot_curve(x_raw, y_raw, fit_method, config)
                
                dec_type = p_cfg.get("decimation")
                mask = np.ones(len(x_raw), dtype=bool)
                shift = 0.0
                
                if opt != "FCI":
                    my_group = None
                    group_idx = 0
                    for i, g in enumerate(groups):
                        if opt in g:
                            my_group = g
                            group_idx = i
                            break
                    if my_group is None:
                        my_group = [opt]
                        group_idx = 0
                        
                    group_size = len(my_group)
                    idx_in_group = my_group.index(opt)
                    
                    x_jitter = get_param(p_cfg.get("x_jitter", 0.0), group_idx)
                    if x_jitter > 0 and group_size > 1:
                        shift = (idx_in_group - (group_size - 1) / 2.0) * x_jitter
                    
                    stride = get_param(p_cfg.get("stride", 3), group_idx)
                    thresh = get_param(p_cfg.get("density_threshold", 0.05), group_idx)
                    sparse_stride = get_param(p_cfg.get("sparse_stride", 4), group_idx)
                    
                    if dec_type == "interleaved":
                        mask = (np.arange(len(x_raw)) % stride) == (idx_in_group % stride)
                    elif dec_type == "density":
                        dy = np.abs(np.gradient(y_raw))
                        mask = (dy > thresh) | ((np.arange(len(x_raw)) - idx_in_group) % sparse_stride == 0)

                lw = 2.5 if opt == "FCI" else config["line_width"] 
                ax.plot(x_curve, y_curve, color=c, linestyle=ls, lw=lw, alpha=alpha_val, zorder=2)
                
                if show_mk: 
                    ax.scatter(x_raw[mask] + shift, y_raw[mask], color=c, marker=mk, 
                               s=ms, edgecolors=config["outline_color"], 
                               linewidths=config["outline_width"], zorder=3, alpha=alpha_val)
                    
                if not legend_built and row_idx == 0 and col_idx == 0:
                    proxy = Line2D([0], [0], color=c, label=opt, marker=mk if show_mk and opt != "FCI" else None, 
                                   markersize=8, linewidth=2, linestyle=ls)
                    legend_handles.append(proxy)
                    
            if panel == "energy_zoomed" and fci_min_x is not None:
                if config.get("show_fci_min_line", False):
                    ax.axvline(x=fci_min_x, color='black', linestyle=':', alpha=0.5, zorder=1)
                if config.get("show_fci_star", False):
                    ax.scatter(fci_min_x, fci_min_y, color='red', marker='*', 
                               s=config.get("marker_size", 50) * 6, edgecolors='none', zorder=10)

            ax.set_title(f"({chr(97 + letter_idx)})", loc='center')
            letter_idx += 1
            
            ax.set_xlabel(r"Distance $d$ ($\AA$)")
            ax.grid(False)
            
            if panel == "energy_raw":
                ax.set_ylabel("Energy (Ha)")
                ax.set_xlim(config.get(f"xlim_energy_raw_{method_suffix}", (0.25, 2.6)))
                ax.set_ylim(config.get(f"ylim_energy_raw_{method_suffix}", (-1.2, -0.4)))
            elif panel == "energy_zoomed":
                ax.set_ylabel("Energy (Ha)")
                ax.set_xlim(config.get(f"xlim_energy_zoomed_{method_suffix}", (0.55, 0.95)))
                ax.set_ylim(config.get(f"ylim_energy_zoomed_{method_suffix}", (-1.145, -1.100)))
            elif panel == "iterations":
                ax.set_ylabel("Iterations (nit)")
                ax.set_xlim(config.get(f"xlim_energy_raw_{method_suffix}", (0.25, 2.6)))
            elif panel == "evaluations":
                ax.set_ylabel("Circuit Executions (nfev)")
                ax.set_xlim(config.get(f"xlim_energy_raw_{method_suffix}", (0.25, 2.6)))

            y_decimals = config.get("y_decimals", {}).get(panel_id)
            if y_decimals is not None:
                ax.yaxis.set_major_formatter(ticker.FormatStrFormatter(f'%.{y_decimals}f'))
                
            x_decimals = config.get("x_decimals", {}).get(panel_id)
            if x_decimals is not None:
                ax.xaxis.set_major_formatter(ticker.FormatStrFormatter(f'%.{x_decimals}f'))

        legend_built = True 

    fig.legend(handles=legend_handles, loc=config["legend_loc"], bbox_to_anchor=config["legend_bbox"], 
               ncol=len(optimizers)+1, frameon=True, fancybox=True, edgecolor='black')

    if save_image:
        save_png(save_image, fig)
        save_pdf(save_image, fig)

    plt.show()

In [ ]:
ENABLE_ADVANCED_VISUALS = True

VISUAL_ENHANCEMENTS = {
    "energy_raw_exact": {
        "show_markers": True,
        "line_styles": {"COBYLA": "-", "SPSA": "--", "SLSQP": ":"} if False else None,
        "marker_hierarchy": {"COBYLA": 140, "SPSA": 80, "SLSQP": 30} if False else None,
        "alpha_hierarchy": {"COBYLA": 0.5, "SPSA": 0.75, "SLSQP": 1.0} if False else None,
        
        "x_jitter": 0.0,
        "groups": [["AQGD"], ["SPSA", "Nelder-Mead", "SLSQP", "COBYLA"]], 
        
        "decimation": "interleaved", 
        "stride": [1, 3], 
        "density_threshold": 0.05, 
        "sparse_stride": 4
    },
    "energy_zoomed_exact": {
        "show_markers": True,
        "line_styles": {"COBYLA": "-", "SPSA": "--", "SLSQP": ":"} if False else None,
        "marker_hierarchy": {"COBYLA": 140, "SPSA": 80, "SLSQP": 30} if False else None,
        "alpha_hierarchy": {"COBYLA": 0.5, "SPSA": 0.75, "SLSQP": 1.0} if False else None,
        "x_jitter": 0.0, "groups": [],
        "decimation": None,
        "stride": 4, "density_threshold": 0.05, "sparse_stride": 4
    },
    "iterations_exact": {
        "show_markers": True,
        "line_styles": {"COBYLA": "-", "SPSA": "--", "SLSQP": ":"} if False else None,
        "marker_hierarchy": {"COBYLA": 140, "SPSA": 80, "SLSQP": 30} if False else None,
        "alpha_hierarchy": {"COBYLA": 0.5, "SPSA": 0.75, "SLSQP": 1.0} if False else None,
        "x_jitter": 0.0, "groups": [],
        "decimation": None, 
        "stride": 4, "density_threshold": 10.0, "sparse_stride": 4
    },
    "evaluations_exact": {
        "show_markers": True,
        "line_styles": {"COBYLA": "-", "SPSA": "--", "SLSQP": ":"} if False else None,
        "marker_hierarchy": {"COBYLA": 140, "SPSA": 80, "SLSQP": 30} if False else None,
        "alpha_hierarchy": {"COBYLA": 0.5, "SPSA": 0.75, "SLSQP": 1.0} if False else None,
        "x_jitter": 0.0, "groups": [],
        "decimation": None,
        "stride": 4, "density_threshold": 50.0, "sparse_stride": 4
    },
    
    "energy_raw_sampling": {
        "show_markers": True,
        "line_styles": {"COBYLA": "-", "SPSA": "--", "SLSQP": ":"} if False else None,
        "marker_hierarchy": {"COBYLA": 140, "SPSA": 80, "SLSQP": 30} if False else None,
        "alpha_hierarchy": {"COBYLA": 0.5, "SPSA": 0.75, "SLSQP": 1.0} if False else None,
        "x_jitter": 0.0, 
        "groups": [["AQGD", "SLSQP"], ["SPSA", "Nelder-Mead", "COBYLA"]],
        "decimation": "interleaved",
        "stride": [2, 3], 
        "density_threshold": 0.05, 
        "sparse_stride": 4
    },
    "energy_zoomed_sampling": {
        "show_markers": True,
        "line_styles": {"COBYLA": "-", "SPSA": "--", "SLSQP": ":"} if False else None,
        "marker_hierarchy": {"COBYLA": 140, "SPSA": 80, "SLSQP": 30} if False else None,
        "alpha_hierarchy": {"COBYLA": 0.5, "SPSA": 0.75, "SLSQP": 1.0} if False else None,
        "x_jitter": 0.0, "groups": [],
        "decimation": None,
        "stride": 4, "density_threshold": 0.05, "sparse_stride": 4
    },
    "iterations_sampling": {
        "show_markers": True,
        "line_styles": {"COBYLA": "-", "SPSA": "--", "SLSQP": ":"} if False else None,
        "marker_hierarchy": {"COBYLA": 140, "SPSA": 80, "SLSQP": 30} if False else None,
        "alpha_hierarchy": {"COBYLA": 0.5, "SPSA": 0.75, "SLSQP": 1.0} if False else None,
        "x_jitter": 0.0, "groups": [],
        "decimation": None,
        "stride": 4, "density_threshold": 10.0, "sparse_stride": 4
    },
    "evaluations_sampling": {
        "show_markers": True,
        "line_styles": {"COBYLA": "-", "SPSA": "--", "SLSQP": ":"} if False else None,
        "marker_hierarchy": {"COBYLA": 140, "SPSA": 80, "SLSQP": 30} if False else None,
        "alpha_hierarchy": {"COBYLA": 0.5, "SPSA": 0.75, "SLSQP": 1.0} if False else None,
        "x_jitter": 0.0, "groups": [],
        "decimation": None,
        "stride": 4, "density_threshold": 50.0, "sparse_stride": 4
    }
}

PLOT_CONFIG_DUAL = {
    "methods": ["aer_exact", "aer_sampling"], 
    
    "panels_row1": ["energy_raw", "energy_zoomed", "iterations"],
    "panels_row2": ["energy_raw", "energy_zoomed", "iterations"],
    
    "fit_methods": {
        "energy_raw_exact": "morse",      
        "energy_zoomed_exact": "morse",  
        "iterations_exact": "raw",      
        "evaluations_exact": "raw",
        "energy_raw_sampling": "morse",      
        "energy_zoomed_sampling": "morse",  
        "iterations_sampling": "raw",      
        "evaluations_sampling": "raw"
    },
    
    "poly_degree": 4,             
    "fit_interval": (0.6, 0.9),   
    
    "xlim_energy_raw_exact": (0.25, 2.6),
    "ylim_energy_raw_exact": (-1.2, -0.4),
    "xlim_energy_zoomed_exact": (0.55, 0.95),
    "ylim_energy_zoomed_exact": (-1.145, -1.100),
    
    "xlim_energy_raw_sampling": (0.25, 2.6),
    "ylim_energy_raw_sampling": (-1.2, 0.1),
    "xlim_energy_zoomed_sampling": (0.55, 0.95),
    "ylim_energy_zoomed_sampling": (-1.145, -1.100),
    
    "y_decimals": {
        "energy_zoomed_exact": 3,
        "energy_zoomed_sampling": 3,
        "energy_raw_exact": 2,
        "energy_raw_sampling": 2
    },
    
    "fig_width": 18,
    "fig_height": 10,
    
    "line_width": 1.5,
    "marker_size": 50,            
    "alpha": 0.85,                
    "outline_width": 0,           
    "outline_color": "none",      
    
    "wspace": 0.35,  
    "hspace": 0.35,  
    
    "legend_loc": "upper center",
    "legend_bbox": (0.5, 0.99),  
    "markers": ['o', 's', '^', 'D', 'X', 'P', 'v', '*'],
    
    "show_fci_min_line": True,    
    "show_fci_star": True
}

plot_optimizer_comparison_dual(
    df_opts, 
    config=PLOT_CONFIG_DUAL, 
    visuals=VISUAL_ENHANCEMENTS,
    use_visuals=ENABLE_ADVANCED_VISUALS,
    #save_image="h2_optimizer_aer_dual_final_v3"
)

In [ ]:
df_opts["Optimizer"].unique()

# 4.

In [ ]:
distances = np.sort(np.concatenate([np.linspace(0.3, 1.0, 15), np.linspace(1.1, 2.5, 5)]))

records = []

method_data = load_data("h2_method_comparison.pkl")
if method_data and "PySCF_FCI" in method_data:
    fci_list = method_data["PySCF_FCI"]
    for i in range(min(len(fci_list), len(distances))):
        if fci_list[i] is not None and not np.isnan(fci_list[i]):
            for m in ["aer_exact", "aer_sampling"]:
                records.append({
                    "Method": m, "Basis": "FCI", "Distance": distances[i], 
                    "Energy": fci_list[i], "Time": np.nan, "Evaluations": np.nan, "Iterations": np.nan
                })

raw_data = load_data("h2_basis_set_comparison_exact_Aer.pkl")
cc_part1 = load_data("h2_basis_set_comparison_exact_cc-pvdz_PART1.pkl")
cc_part2 = load_data("h2_basis_set_comparison_exact_cc-pvdz_PART2.pkl")

list_part1 = cc_part1.get("cc-pvdz", cc_part1) if isinstance(cc_part1, dict) else cc_part1
list_part2 = cc_part2.get("cc-pvdz", cc_part2) if isinstance(cc_part2, dict) else cc_part2

if raw_data is not None and list_part1 is not None and list_part2 is not None:
    raw_data["cc-pvdz"] = list_part1 + list_part2

if raw_data:
    for basis, data_list in raw_data.items():
        for i, item in enumerate(data_list):
            if item is None: continue
            dist = distances[i] if i < len(distances) else distances[-1]
            energy, elapsed, metrics = item
            
            evals = np.nan
            iters = np.nan
            if isinstance(metrics, dict):
                evals = metrics.get('cost_function_evals', metrics.get('nfev', np.nan))
                iters = metrics.get('nit', np.nan)
            
            records.append({
                "Method": "aer_exact", 
                "Basis": basis, 
                "Distance": dist,
                "Energy": energy,
                "Time": elapsed,
                "Evaluations": evals,
                "Iterations": iters
            })

df_basis = pd.DataFrame(records)

In [ ]:
def plot_basis_analysis(df, method_filter, config, visuals, use_visuals=False, save_image=False):
    subset = df[df["Method"] == method_filter].copy()
    if subset.empty: return
    
    basis_sets = [b for b in subset["Basis"].unique() if b != "FCI"]
    include_fci = config.get("include_fci_baseline", True)
    
    palette = sns.color_palette("tab10", n_colors=len(basis_sets))
    color_map = dict(zip(basis_sets, palette))
    color_map["FCI"] = "#000000"
    
    base_marker_map = dict(zip(basis_sets, config["markers"][:len(basis_sets)]))
    base_marker_map["FCI"] = ""

    panels = config.get("panels", [])
    ncols = len(panels)
    if ncols == 0: return
    
    fig_w = config.get("fig_width", 18)
    fig_h = config.get("fig_height", 5)
    fig, axes = plt.subplots(1, ncols, figsize=(fig_w, fig_h))
    
    if ncols == 1: axes = [axes]
        
    plt.subplots_adjust(wspace=config["wspace"])

    def morse_func(r, De, a, re, E_shift):
        return De * (1 - np.exp(-a * (r - re)))**2 + E_shift

    def get_plot_curve(x, y, fit_type, cfg):
        idx = np.argsort(x)
        x, y = x[idx], y[idx]
        
        if fit_type == "raw": return x, y
        x_dense = np.linspace(min(x), max(x), 500)
        
        if fit_type == "spline":
            try: 
                x_clean, unique_idx = np.unique(x, return_index=True)
                y_clean = y[unique_idx]
                if len(x_clean) > 3: return x_dense, make_interp_spline(x_clean, y_clean, k=3)(x_dense)
                return x, y
            except: return x, y
            
        elif fit_type == "morse":
            try:
                mask = x > 0.45
                popt, _ = curve_fit(morse_func, x[mask], y[mask], p0=[0.2, 1.5, 0.74, -1.137], maxfev=20000)
                return x_dense, morse_func(x_dense, *popt)
            except: return x, y
            
        elif fit_type == "poly":
            interval = cfg.get("fit_interval", (0.5, 1.0))
            mask = (x >= interval[0]) & (x <= interval[1])
            if len(x[mask]) > cfg.get("poly_degree", 4):
                coeffs = np.polyfit(x[mask], y[mask], cfg.get("poly_degree", 4))
                x_local = np.linspace(interval[0], interval[1], 300)
                return x_local, np.poly1d(coeffs)(x_local)
            return x, y
            
        return x, y 

    def get_param(param_val, g_idx):
        if isinstance(param_val, list):
            return param_val[g_idx] if g_idx < len(param_val) else param_val[-1]
        return param_val

    fci_min_x, fci_min_y = None, None
    if include_fci:
        fci_data_df = subset[subset["Basis"] == "FCI"].sort_values("Distance")
        fci_dist = fci_data_df["Distance"].values
        fci_energies = fci_data_df["Energy"].values
        if len(fci_dist) > 0:
            zoom_fit = config.get("fit_methods", {}).get("energy_zoomed", "spline")
            f_x, f_y = get_plot_curve(fci_dist, fci_energies, zoom_fit, config)
            f_min_idx = np.argmin(f_y)
            fci_min_x, fci_min_y = f_x[f_min_idx], f_y[f_min_idx]

    legend_handles = []

    for idx, panel in enumerate(panels):
        ax = axes[idx]
        p_cfg = visuals.get(panel, {}) if use_visuals else {}
        fit_method = config.get("fit_methods", {}).get(panel, "raw")
        
        y_col = "Energy"
        if panel == "time": y_col = "Time"
        elif panel == "iterations": y_col = "Iterations"
        elif panel == "evaluations": y_col = "Evaluations"
        
        opts_to_plot = ["FCI"] + basis_sets if (panel in ["energy_raw", "energy_zoomed"] and include_fci) else basis_sets
        
        groups = p_cfg.get("groups", [])
        if not groups: groups = [basis_sets] 
        
        for b_idx, basis in enumerate(opts_to_plot):
            d = subset[subset["Basis"] == basis].sort_values("Distance").dropna(subset=[y_col])
            
            if panel == "time" and config.get("log_time", True):
                d = d[d[y_col] > 0]
                
            if len(d) == 0: continue
            
            c = color_map.get(basis, "grey")
            mk = base_marker_map.get(basis, "o")
            
            ls_dict = p_cfg.get("line_styles")
            ls = ls_dict.get(basis, "-" if basis != "FCI" else "--") if ls_dict else ("-" if basis != "FCI" else "--")
            show_mk = p_cfg.get("show_markers", True) and basis != "FCI"
            
            show_outline = p_cfg.get("show_outline", False)
            edge_c = config.get("outline_color", "white") if show_outline else "none"
            line_w = config.get("outline_width", 1.0) if show_outline else 0.0
            
            ms = config.get("marker_size", 50)
            mh_dict = p_cfg.get("marker_hierarchy")
            if mh_dict is not None: ms = mh_dict.get(basis, ms)
            if panel == "energy_zoomed": ms *= 1.5
            
            alpha_val = config.get("alpha", 0.85)
            ah_dict = p_cfg.get("alpha_hierarchy")
            if ah_dict is not None and basis != "FCI": alpha_val = ah_dict.get(basis, alpha_val)
            if basis == "FCI": alpha_val = config.get("alpha", 0.85)
            
            x_raw, y_raw = d["Distance"].values, d[y_col].values
            x_curve, y_curve = get_plot_curve(x_raw, y_raw, fit_method, config)
            
            dec_type = p_cfg.get("decimation")
            mask = np.ones(len(x_raw), dtype=bool)
            shift = 0.0
            
            if basis != "FCI":
                my_group = next((g for g in groups if basis in g), [basis])
                group_idx = groups.index(my_group) if my_group in groups else 0
                group_size = len(my_group)
                idx_in_group = my_group.index(basis)
                
                x_jitter = get_param(p_cfg.get("x_jitter", 0.0), group_idx)
                if x_jitter > 0 and group_size > 1:
                    shift = (idx_in_group - (group_size - 1) / 2.0) * x_jitter
                
                stride = get_param(p_cfg.get("stride", 3), group_idx)
                thresh = get_param(p_cfg.get("density_threshold", 0.05), group_idx)
                sparse_stride = get_param(p_cfg.get("sparse_stride", 4), group_idx)
                
                if dec_type == "interleaved":
                    mask = (np.arange(len(x_raw)) % stride) == (idx_in_group % stride)
                elif dec_type == "density":
                    dy = np.abs(np.gradient(y_raw))
                    mask = (dy > thresh) | ((np.arange(len(x_raw)) - idx_in_group) % sparse_stride == 0)

            lw = 2.5 if basis == "FCI" else config["line_width"] 
            ax.plot(x_curve, y_curve, color=c, linestyle=ls, lw=lw, alpha=alpha_val, zorder=2)
            
            if show_mk: 
                ax.scatter(x_raw[mask] + shift, y_raw[mask], color=c, marker=mk, 
                           s=ms, edgecolors=edge_c, 
                           linewidths=line_w, zorder=3, alpha=alpha_val)
                
        if idx == 0:
            legend_basis = ["FCI"] + basis_sets if include_fci else basis_sets
            for basis in legend_basis:
                c = color_map.get(basis, "grey")
                ls_dict = p_cfg.get("line_styles")
                ls = ls_dict.get(basis, "-" if basis != "FCI" else "--") if ls_dict else ("-" if basis != "FCI" else "--")
                show_mk = p_cfg.get("show_markers", True) and basis != "FCI"
                mk = base_marker_map.get(basis, "o") if show_mk else None
                proxy = Line2D([0], [0], color=c, label=basis.upper() if basis != "FCI" else "FCI", 
                               marker=mk, markersize=8, linewidth=2, linestyle=ls)
                legend_handles.append(proxy)
                    
        if panel == "energy_zoomed" and include_fci and fci_min_x is not None:
            if config.get("show_fci_min_line", False):
                ax.axvline(x=fci_min_x, color='black', linestyle=':', alpha=0.5, zorder=1)
            if config.get("show_fci_star", False):
                ax.scatter(fci_min_x, fci_min_y, color='red', marker='*', 
                           s=config.get("marker_size", 50) * 6, edgecolors='none', zorder=10)

        ax.set_title(f"({chr(97 + idx)})", loc='center')
        ax.set_xlabel(r"Distance $d$ ($\AA$)")
        ax.grid(False)
        
        if panel == "energy_raw":
            ax.set_ylabel("Energy (Ha)")
            ax.set_xlim(config.get("xlim_energy_raw", (0.25, 2.6)))
            ax.set_ylim(config.get("ylim_energy_raw", (-1.2, -0.4)))
        elif panel == "energy_zoomed":
            ax.set_ylabel("Energy (Ha)")
            ax.set_xlim(config.get("xlim_energy_zoomed", (0.50, 1.0)))
            ax.set_ylim(config.get("ylim_energy_zoomed", (-1.16, -1.02)))
        elif panel == "time":
            ax.set_ylabel("Time (s)")
            ax.set_xlim(config.get("xlim_energy_raw", (0.25, 2.6)))
            if config.get("log_time", True):
                ax.set_yscale("log")
                ax.yaxis.set_major_locator(ticker.LogLocator(base=10.0, numticks=10))
                ax.yaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs=(0.2, 0.4, 0.6, 0.8), numticks=10))
                ax.yaxis.set_minor_formatter(ticker.NullFormatter())
        elif panel == "iterations":
            ax.set_ylabel("Iterations (nit)")
            ax.set_xlim(config.get("xlim_energy_raw", (0.25, 2.6)))
        elif panel == "evaluations":
            ax.set_ylabel("Circuit Executions (nfev)")
            ax.set_xlim(config.get("xlim_energy_raw", (0.25, 2.6)))

        if panel != "time" or not config.get("log_time", True):
            y_decimals = config.get("y_decimals", {}).get(panel)
            if y_decimals is not None:
                ax.yaxis.set_major_formatter(ticker.FormatStrFormatter(f'%.{y_decimals}f'))
                
            x_decimals = config.get("x_decimals", {}).get(panel)
            if x_decimals is not None:
                ax.xaxis.set_major_formatter(ticker.FormatStrFormatter(f'%.{x_decimals}f'))

    fig.legend(handles=legend_handles, loc=config["legend_loc"], bbox_to_anchor=config["legend_bbox"], 
               ncol=len(legend_handles), frameon=True, fancybox=True, edgecolor='black')

    if save_image:
        save_png(save_image, fig)
        save_pdf(save_image, fig)

    plt.show()

In [ ]:
ENABLE_ADVANCED_VISUALS = True

VISUAL_ENHANCEMENTS = {
    "energy_raw": {
        "show_markers": True,
        "show_outline": False, 
        "line_styles": {"sto-3g": "-", "sto-6g": "--", "6-31g": "-.", "6-311g": ":", "cc-pvdz": "-"} if False else None,
        "marker_hierarchy": {"sto-3g": 100, "sto-6g": 80, "6-31g": 60, "6-311g": 40, "cc-pvdz": 20} if False else None,
        "alpha_hierarchy": {"sto-3g": 0.5, "sto-6g": 0.6, "6-31g": 0.7, "6-311g": 0.85, "cc-pvdz": 1.0} if False else None,
        
        "x_jitter": 0.0,
        "groups": [["sto-3g", "sto-6g"], ["6-31g", "6-311g", "cc-pvdz"]], 
        "decimation": "interleaved", 
        "stride": [2, 3], 
        "density_threshold": 0.05, 
        "sparse_stride": 4
    },
    
    "energy_zoomed": {
        "show_markers": True,
        "show_outline": False,
        "line_styles": {"sto-3g": "-", "sto-6g": "--", "6-31g": "-.", "6-311g": ":", "cc-pvdz": "-"} if False else None,
        "marker_hierarchy": {"sto-3g": 100, "sto-6g": 80, "6-31g": 60, "6-311g": 40, "cc-pvdz": 20} if False else None,
        "alpha_hierarchy": {"sto-3g": 0.5, "sto-6g": 0.6, "6-31g": 0.7, "6-311g": 0.85, "cc-pvdz": 1.0} if False else None,
        "x_jitter": 0.0, "groups": [],
        "decimation": None,
        "stride": 4, "density_threshold": 0.05, "sparse_stride": 4
    },
    
    "time": {
        "show_markers": True,
        "show_outline": False,
        "line_styles": {"sto-3g": "-", "sto-6g": "--", "6-31g": "-.", "6-311g": ":", "cc-pvdz": "-"} if False else None,
        "marker_hierarchy": {"sto-3g": 100, "sto-6g": 80, "6-31g": 60, "6-311g": 40, "cc-pvdz": 20} if False else None,
        "alpha_hierarchy": {"sto-3g": 0.5, "sto-6g": 0.6, "6-31g": 0.7, "6-311g": 0.85, "cc-pvdz": 1.0} if False else None,
        "x_jitter": 0.0, "groups": [],
        "decimation": None, 
        "stride": 4, "density_threshold": 1.0, "sparse_stride": 4
    },
    
    "iterations": {
        "show_markers": True,
        "show_outline": False,
        "line_styles": {"sto-3g": "-", "sto-6g": "--", "6-31g": "-.", "6-311g": ":", "cc-pvdz": "-"} if False else None,
        "marker_hierarchy": {"sto-3g": 100, "sto-6g": 80, "6-31g": 60, "6-311g": 40, "cc-pvdz": 20} if False else None,
        "alpha_hierarchy": {"sto-3g": 0.5, "sto-6g": 0.6, "6-31g": 0.7, "6-311g": 0.85, "cc-pvdz": 1.0} if False else None,
        "x_jitter": 0.0, "groups": [],
        "decimation": None, 
        "stride": 4, "density_threshold": 10.0, "sparse_stride": 4
    },
    
    "evaluations": {
        "show_markers": True,
        "show_outline": False,
        "line_styles": {"sto-3g": "-", "sto-6g": "--", "6-31g": "-.", "6-311g": ":", "cc-pvdz": "-"} if False else None,
        "marker_hierarchy": {"sto-3g": 100, "sto-6g": 80, "6-31g": 60, "6-311g": 40, "cc-pvdz": 20} if False else None,
        "alpha_hierarchy": {"sto-3g": 0.5, "sto-6g": 0.6, "6-31g": 0.7, "6-311g": 0.85, "cc-pvdz": 1.0} if False else None,
        "x_jitter": 0.0, "groups": [],
        "decimation": None,
        "stride": 4, "density_threshold": 50.0, "sparse_stride": 4
    }
}

PLOT_CONFIG_BASIS = {

    "panels": ["energy_raw", "energy_zoomed", "evaluations"],
    
    "fit_methods": {
        "energy_raw": "morse",      
        "energy_zoomed": "morse",  
        "time": "raw",      
        "iterations": "raw",      
        "evaluations": "raw"
    },
    
    "include_fci_baseline": False,
    "show_fci_min_line": False,    
    "show_fci_star": False,
    
    "poly_degree": 4,             
    "fit_interval": (0.6, 0.9),   
    
    "global_xlim": (0.25, 2.6),
    "xlim_energy_raw": (0.25, 2.6),
    "ylim_energy_raw": (-1.2, -0.4),
    
    "xlim_energy_zoomed": (0.55, 0.98),
    "ylim_energy_zoomed": (-1.17, -1.100),
    
    "log_time": True,
    
    "y_decimals": {
        "energy_raw": 2,
        "energy_zoomed": 3
    },
    
    "fig_width": 18,
    "fig_height": 5,
    
    "line_width": 1.5,
    "marker_size": 60,            
    "alpha": 0.85,                
    "outline_width": 1.0,           
    "outline_color": "white",      
    
    "wspace": 0.35,  
    
    "legend_loc": "upper center",
    "legend_bbox": (0.5, 1.08),  
    "markers": ['o', 's', '^', 'D', 'X', 'P', 'v', '*']
}

plot_basis_analysis(
    df_basis, 
    method_filter="aer_exact", 
    config=PLOT_CONFIG_BASIS, 
    visuals=VISUAL_ENHANCEMENTS,
    use_visuals=ENABLE_ADVANCED_VISUALS,
    save_image="h2_basis_set_comparison_aer_exact_final_v3"
)

# 5. 

# 6. 

In [ ]:
filename = "h2_noise_comparison.pkl"
raw_data = load_data(filename) 

distances = np.concatenate([np.linspace(0.3, 1.0, 15), np.linspace(1.1, 2.5, 5)])
distances = np.sort(distances)

records = []
if raw_data is not None:
    for noise_model, data_list in raw_data.items():
        for i, (energy, elapsed, metrics) in enumerate(data_list):
            dist = distances[i]
            
            evals = metrics.get('cost_function_evals', metrics.get('nfev', 0))
            iterations = metrics.get('nit', 0)
            
            records.append({
                "Distance": dist,
                "Method": "aer_sampling", 
                "Noise_Model": noise_model,
                "Energy": energy,
                "Time": elapsed,
                "Evaluations": evals,
                "Iterations": iterations
            })
            
df_noise = pd.DataFrame(records)

In [ ]:
def plot_noise_analysis(df, method_filter, config, visuals, use_visuals=False, save_image=False):
    subset = df[df["Method"] == method_filter].copy()
    if subset.empty: return
    
    noise_models = [n for n in subset["Noise_Model"].unique() if n != "FCI"]
    include_fci = config.get("include_fci_baseline", True)
    
    palette = sns.color_palette("tab10", n_colors=len(noise_models))
    color_map = dict(zip(noise_models, palette))
    color_map["FCI"] = "#000000"
    
    base_marker_map = dict(zip(noise_models, config["markers"][:len(noise_models)]))
    base_marker_map["FCI"] = ""

    panels = config.get("panels", [])
    ncols = len(panels)
    if ncols == 0: return
    
    fig_w = config.get("fig_width", 18)
    fig_h = config.get("fig_height", 5)
    fig, axes = plt.subplots(1, ncols, figsize=(fig_w, fig_h))
    
    if ncols == 1: axes = [axes]
        
    plt.subplots_adjust(wspace=config["wspace"])

    def morse_func(r, De, a, re, E_shift):
        return De * (1 - np.exp(-a * (r - re)))**2 + E_shift

    def get_plot_curve(x, y, fit_type, cfg):
        idx = np.argsort(x)
        x, y = x[idx], y[idx]
        
        if fit_type == "raw": return x, y
        x_dense = np.linspace(min(x), max(x), 500)
        
        if fit_type == "spline":
            try: 
                x_clean, unique_idx = np.unique(x, return_index=True)
                y_clean = y[unique_idx]
                if len(x_clean) > 3: return x_dense, make_interp_spline(x_clean, y_clean, k=3)(x_dense)
                return x, y
            except: return x, y
            
        elif fit_type == "morse":
            try:
                mask = x > 0.45
                popt, _ = curve_fit(morse_func, x[mask], y[mask], p0=[0.2, 1.5, 0.74, -1.137], maxfev=20000)
                return x_dense, morse_func(x_dense, *popt)
            except: return x, y
            
        elif fit_type == "poly":
            interval = cfg.get("fit_interval", (0.5, 1.0))
            mask = (x >= interval[0]) & (x <= interval[1])
            if len(x[mask]) > cfg.get("poly_degree", 4):
                coeffs = np.polyfit(x[mask], y[mask], cfg.get("poly_degree", 4))
                x_local = np.linspace(interval[0], interval[1], 300)
                return x_local, np.poly1d(coeffs)(x_local)
            return x, y
            
        return x, y 

    def get_param(param_val, g_idx):
        if isinstance(param_val, list):
            return param_val[g_idx] if g_idx < len(param_val) else param_val[-1]
        return param_val

    fci_min_x, fci_min_y = None, None
    if include_fci:
        fci_data_df = subset[subset["Noise_Model"] == "FCI"].sort_values("Distance")
        fci_dist = fci_data_df["Distance"].values
        fci_energies = fci_data_df["Energy"].values
        if len(fci_dist) > 0:
            zoom_fit = config.get("fit_methods", {}).get("energy_zoomed", "spline")
            f_x, f_y = get_plot_curve(fci_dist, fci_energies, zoom_fit, config)
            f_min_idx = np.argmin(f_y)
            fci_min_x, fci_min_y = f_x[f_min_idx], f_y[f_min_idx]

    legend_handles = []

    for idx, panel in enumerate(panels):
        ax = axes[idx]
        p_cfg = visuals.get(panel, {}) if use_visuals else {}
        fit_method = config.get("fit_methods", {}).get(panel, "raw")
        
        y_col = "Energy"
        if panel == "time": y_col = "Time"
        elif panel == "iterations": y_col = "Iterations"
        elif panel == "evaluations": y_col = "Evaluations"
        
        opts_to_plot = ["FCI"] + noise_models if (panel in ["energy_raw", "energy_zoomed"] and include_fci) else noise_models
        
        groups = p_cfg.get("groups", [])
        if not groups: groups = [noise_models] 
        
        for b_idx, n_model in enumerate(opts_to_plot):
            d = subset[subset["Noise_Model"] == n_model].sort_values("Distance").dropna(subset=[y_col])
            
            if panel == "time" and config.get("log_time", True):
                d = d[d[y_col] > 0]
                
            if len(d) == 0: continue
            
            c = color_map.get(n_model, "grey")
            mk = base_marker_map.get(n_model, "o")
            
            ls_dict = p_cfg.get("line_styles")
            ls = ls_dict.get(n_model, "-" if n_model != "FCI" else "--") if ls_dict else ("-" if n_model != "FCI" else "--")
            show_mk = p_cfg.get("show_markers", True) and n_model != "FCI"
            
            show_outline = p_cfg.get("show_outline", False)
            edge_c = config.get("outline_color", "white") if show_outline else "none"
            line_w = config.get("outline_width", 1.0) if show_outline else 0.0
            
            ms = config.get("marker_size", 50)
            mh_dict = p_cfg.get("marker_hierarchy")
            if mh_dict is not None: ms = mh_dict.get(n_model, ms)
            if panel == "energy_zoomed": ms *= 1.5
            
            alpha_val = config.get("alpha", 0.85)
            ah_dict = p_cfg.get("alpha_hierarchy")
            if ah_dict is not None and n_model != "FCI": alpha_val = ah_dict.get(n_model, alpha_val)
            if n_model == "FCI": alpha_val = config.get("alpha", 0.85)
            
            x_raw, y_raw = d["Distance"].values, d[y_col].values
            x_curve, y_curve = get_plot_curve(x_raw, y_raw, fit_method, config)
            
            dec_type = p_cfg.get("decimation")
            mask = np.ones(len(x_raw), dtype=bool)
            shift = 0.0
            
            if n_model != "FCI":
                my_group = next((g for g in groups if n_model in g), [n_model])
                group_idx = groups.index(my_group) if my_group in groups else 0
                group_size = len(my_group)
                idx_in_group = my_group.index(n_model)
                
                x_jitter = get_param(p_cfg.get("x_jitter", 0.0), group_idx)
                if x_jitter > 0 and group_size > 1:
                    shift = (idx_in_group - (group_size - 1) / 2.0) * x_jitter
                
                stride = get_param(p_cfg.get("stride", 3), group_idx)
                thresh = get_param(p_cfg.get("density_threshold", 0.05), group_idx)
                sparse_stride = get_param(p_cfg.get("sparse_stride", 4), group_idx)
                
                if dec_type == "interleaved":
                    mask = (np.arange(len(x_raw)) % stride) == (idx_in_group % stride)
                elif dec_type == "density":
                    dy = np.abs(np.gradient(y_raw))
                    mask = (dy > thresh) | ((np.arange(len(x_raw)) - idx_in_group) % sparse_stride == 0)

            lw = 2.5 if n_model == "FCI" else config["line_width"] 
            ax.plot(x_curve, y_curve, color=c, linestyle=ls, lw=lw, alpha=alpha_val, zorder=2)
            
            if show_mk: 
                ax.scatter(x_raw[mask] + shift, y_raw[mask], color=c, marker=mk, 
                           s=ms, edgecolors=edge_c, 
                           linewidths=line_w, zorder=3, alpha=alpha_val)
                
        if idx == 0:
            legend_models = ["FCI"] + noise_models if include_fci else noise_models
            for n_model in legend_models:
                c = color_map.get(n_model, "grey")
                ls_dict = p_cfg.get("line_styles")
                ls = ls_dict.get(n_model, "-" if n_model != "FCI" else "--") if ls_dict else ("-" if n_model != "FCI" else "--")
                show_mk = p_cfg.get("show_markers", True) and n_model != "FCI"
                mk = base_marker_map.get(n_model, "o") if show_mk else None
                
                if n_model == "Ideal_Shot_Noise":
                    label_str = "VQE Sampling"
                elif n_model == "FCI":
                    label_str = "FCI"
                else:
                    label_str = n_model.replace("_", " ")
                    
                proxy = Line2D([0], [0], color=c, label=label_str, 
                               marker=mk, markersize=8, linewidth=2, linestyle=ls)
                legend_handles.append(proxy)
                    
        if panel == "energy_zoomed" and include_fci and fci_min_x is not None:
            if config.get("show_fci_min_line", False):
                ax.axvline(x=fci_min_x, color='black', linestyle=':', alpha=0.5, zorder=1)
            if config.get("show_fci_star", False):
                ax.scatter(fci_min_x, fci_min_y, color='red', marker='*', 
                           s=config.get("marker_size", 50) * 6, edgecolors='none', zorder=10)

        ax.set_title(f"({chr(97 + idx)})", loc='center')
        ax.set_xlabel(r"Distance $d$ ($\AA$)")
        ax.grid(False)
        
        if panel == "energy_raw":
            ax.set_ylabel("Energy (Ha)")
            ax.set_xlim(config.get("xlim_energy_raw", (0.25, 2.6)))
            ax.set_ylim(config.get("ylim_energy_raw", (-1.2, -0.4)))
        elif panel == "energy_zoomed":
            ax.set_ylabel("Energy (Ha)")
            ax.set_xlim(config.get("xlim_energy_zoomed", (0.50, 1.0)))
            ax.set_ylim(config.get("ylim_energy_zoomed", (-1.16, -1.02)))
        elif panel == "time":
            ax.set_ylabel("Time (s)")
            ax.set_xlim(config.get("xlim_energy_raw", (0.25, 2.6)))
            if config.get("log_time", True):
                ax.set_yscale("log")
                ax.yaxis.set_major_locator(ticker.LogLocator(base=10.0, numticks=10))
                ax.yaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs=(0.2, 0.4, 0.6, 0.8), numticks=10))
                ax.yaxis.set_minor_formatter(ticker.NullFormatter())
        elif panel == "iterations":
            ax.set_ylabel("Iterations (nit)")
            ax.set_xlim(config.get("xlim_energy_raw", (0.25, 2.6)))
        elif panel == "evaluations":
            ax.set_ylabel("Circuit Executions (nfev)")
            ax.set_xlim(config.get("xlim_energy_raw", (0.25, 2.6)))

        if panel != "time" or not config.get("log_time", True):
            y_decimals = config.get("y_decimals", {}).get(panel)
            if y_decimals is not None:
                ax.yaxis.set_major_formatter(ticker.FormatStrFormatter(f'%.{y_decimals}f'))
                
            x_decimals = config.get("x_decimals", {}).get(panel)
            if x_decimals is not None:
                ax.xaxis.set_major_formatter(ticker.FormatStrFormatter(f'%.{x_decimals}f'))

    default_ncol = int(np.ceil(len(legend_handles) / 2.0))
    fig.legend(handles=legend_handles, 
               loc=config.get("legend_loc", "lower center"), 
               bbox_to_anchor=config.get("legend_bbox", (0.5, 1.02)), 
               ncol=config.get("legend_ncol", default_ncol), 
               frameon=True, fancybox=True, edgecolor='black')

    if save_image:
        save_png(save_image, fig)
        save_pdf(save_image, fig)

    plt.show()

In [ ]:
ENABLE_ADVANCED_VISUALS = True

VISUAL_ENHANCEMENTS = {
    "energy_raw": {
        "show_markers": True,
        "show_outline": False, 
        "line_styles": {"Ideal_Shot_Noise": "-", "Fake_Manila_5Q": "--", "Fake_Jakarta_7Q": "--", "Fake_Torino_133Q": "-.", "Fake_Kyoto_127Q": "-.", "Fake_Osaka_127Q": "-.", "Fake_Brisbane_127Q": ":"} if False else None,
        "marker_hierarchy": {"Ideal_Shot_Noise": 120, "Fake_Manila_5Q": 90, "Fake_Jakarta_7Q": 90, "Fake_Torino_133Q": 60, "Fake_Kyoto_127Q": 60, "Fake_Osaka_127Q": 60, "Fake_Brisbane_127Q": 60} if False else None,
        "alpha_hierarchy": {"Ideal_Shot_Noise": 1.0, "Fake_Manila_5Q": 0.85, "Fake_Jakarta_7Q": 0.85, "Fake_Torino_133Q": 0.7, "Fake_Kyoto_127Q": 0.7, "Fake_Osaka_127Q": 0.7, "Fake_Brisbane_127Q": 0.7} if False else None,
        
        "x_jitter": 0.0,
        "groups": [["Ideal_Shot_Noise"], ["Fake_Manila_5Q", "Fake_Jakarta_7Q"], ["Fake_Torino_133Q", "Fake_Kyoto_127Q", "Fake_Osaka_127Q", "Fake_Brisbane_127Q"]], 
        "decimation": None, 
        "stride": [1, 2, 4], 
        "density_threshold": 0.05, 
        "sparse_stride": 4
    },
    
    "energy_zoomed": {
        "show_markers": True,
        "show_outline": False,
        "line_styles": {"Ideal_Shot_Noise": "-", "Fake_Manila_5Q": "--", "Fake_Jakarta_7Q": "--", "Fake_Torino_133Q": "-.", "Fake_Kyoto_127Q": "-.", "Fake_Osaka_127Q": "-.", "Fake_Brisbane_127Q": ":"} if False else None,
        "marker_hierarchy": {"Ideal_Shot_Noise": 120, "Fake_Manila_5Q": 90, "Fake_Jakarta_7Q": 90, "Fake_Torino_133Q": 60, "Fake_Kyoto_127Q": 60, "Fake_Osaka_127Q": 60, "Fake_Brisbane_127Q": 60} if False else None,
        "alpha_hierarchy": {"Ideal_Shot_Noise": 1.0, "Fake_Manila_5Q": 0.85, "Fake_Jakarta_7Q": 0.85, "Fake_Torino_133Q": 0.7, "Fake_Kyoto_127Q": 0.7, "Fake_Osaka_127Q": 0.7, "Fake_Brisbane_127Q": 0.7} if False else None,
        "x_jitter": 0.0, "groups": [],
        "decimation": None,
        "stride": 4, "density_threshold": 0.05, "sparse_stride": 4
    },
    
    "time": {
        "show_markers": True,
        "show_outline": False,
        "line_styles": {"Ideal_Shot_Noise": "-", "Fake_Manila_5Q": "--", "Fake_Jakarta_7Q": "--", "Fake_Torino_133Q": "-.", "Fake_Kyoto_127Q": "-.", "Fake_Osaka_127Q": "-.", "Fake_Brisbane_127Q": ":"} if False else None,
        "marker_hierarchy": {"Ideal_Shot_Noise": 120, "Fake_Manila_5Q": 90, "Fake_Jakarta_7Q": 90, "Fake_Torino_133Q": 60, "Fake_Kyoto_127Q": 60, "Fake_Osaka_127Q": 60, "Fake_Brisbane_127Q": 60} if False else None,
        "alpha_hierarchy": {"Ideal_Shot_Noise": 1.0, "Fake_Manila_5Q": 0.85, "Fake_Jakarta_7Q": 0.85, "Fake_Torino_133Q": 0.7, "Fake_Kyoto_127Q": 0.7, "Fake_Osaka_127Q": 0.7, "Fake_Brisbane_127Q": 0.7} if False else None,
        "x_jitter": 0.0, "groups": [],
        "decimation": None, 
        "stride": 4, "density_threshold": 1.0, "sparse_stride": 4
    },
    
    "evaluations": {
        "show_markers": True,
        "show_outline": False,
        "line_styles": {"Ideal_Shot_Noise": "-", "Fake_Manila_5Q": "--", "Fake_Jakarta_7Q": "--", "Fake_Torino_133Q": "-.", "Fake_Kyoto_127Q": "-.", "Fake_Osaka_127Q": "-.", "Fake_Brisbane_127Q": ":"} if False else None,
        "marker_hierarchy": {"Ideal_Shot_Noise": 120, "Fake_Manila_5Q": 90, "Fake_Jakarta_7Q": 90, "Fake_Torino_133Q": 60, "Fake_Kyoto_127Q": 60, "Fake_Osaka_127Q": 60, "Fake_Brisbane_127Q": 60} if False else None,
        "alpha_hierarchy": {"Ideal_Shot_Noise": 1.0, "Fake_Manila_5Q": 0.85, "Fake_Jakarta_7Q": 0.85, "Fake_Torino_133Q": 0.7, "Fake_Kyoto_127Q": 0.7, "Fake_Osaka_127Q": 0.7, "Fake_Brisbane_127Q": 0.7} if False else None,
        "x_jitter": 0.0, "groups": [],
        "decimation": None,
        "stride": 4, "density_threshold": 50.0, "sparse_stride": 4
    }
}

PLOT_CONFIG_NOISE = {
    "panels": ["energy_raw", "energy_zoomed", "time"],
    
    "fit_methods": {
        "energy_raw": "morse",      
        "energy_zoomed": "morse",  
        "time": "raw",      
        "iterations": "raw",      
        "evaluations": "raw"
    },
    
    "include_fci_baseline": False,
    "show_fci_min_line": False,    
    "show_fci_star": False,
    
    "poly_degree": 4,             
    "fit_interval": (0.6, 0.9),   
    
    "global_xlim": (0.25, 2.6),
    "xlim_energy_raw": (0.25, 2.6),
    "ylim_energy_raw": (-1.4, 1.6),
    
    "xlim_energy_zoomed": (0.4, 1.2),
    "ylim_energy_zoomed": (-1.2, 0),
    
    "log_time": True,
    
    "y_decimals": {
        "energy_raw": 2,
        "energy_zoomed": 3
    },
    
    "fig_width": 18,
    "fig_height": 5,
    
    "line_width": 1.5,
    "marker_size": 60,            
    "alpha": 0.85,                
    "outline_width": 1.0,           
    "outline_color": "white",      
    
    "wspace": 0.35,  
    
    "legend_ncol": 4,              
    "legend_loc": "lower center",  
    "legend_bbox": (0.5, 1.0),    
    "markers": ['o', 's', '^', 'D', 'X', 'P', 'v', '*']
}

plot_noise_analysis(
    df_noise, 
    method_filter="aer_sampling", 
    config=PLOT_CONFIG_NOISE, 
    visuals=VISUAL_ENHANCEMENTS,
    use_visuals=ENABLE_ADVANCED_VISUALS,
    save_image="h2_noise_comparison_final_v3"
)